# FoodBridge | Assignment 2: Data Dictionary and Cleaning


Only the six primary tables are profiled. Workbook dictionaries are used as references. The source tables remain unchanged, and the datasets stay separate.

## 1. Select the six Excel files



In [ ]:
from pathlib import Path
import re, unicodedata, difflib, zipfile
import pandas as pd
import numpy as np
from IPython.display import display

try:
    from google.colab import files
    uploaded = files.upload()
    DATA_DIR = Path('/content')
    # If a ZIP is uploaded, copy its Excel workbooks into the Colab folder.
    for filename in uploaded:
        uploaded_path=DATA_DIR/filename
        if uploaded_path.suffix.lower()=='.zip':
            with zipfile.ZipFile(uploaded_path) as archive:
                for member in archive.namelist():
                    if member.lower().endswith(('.xlsx','.xls')):
                        (DATA_DIR/Path(member).name).write_bytes(archive.read(member))
except ImportError:
    DATA_DIR = Path('.')  # Local fallback only when google.colab is unavailable.

SPECS = {
 'National NGO Organizations Updated Synthetic-Data Disclosure': {
   'tokens':['national','ngo','organizations'], 'sheet':'Organizations'},
 'SH Population Service and Demand Mock Dataset': {
   'tokens':['population','service','demand'], 'sheet':'Population Service Demand'},
 'SH Donation Allocations Mock Dataset': {
   'tokens':['donation','allocations'], 'sheet':'Donation Allocations'},
 'SH Organizations Mock Dataset v2': {
   'tokens':['organizations','mock'], 'sheet':'Organizations'},
 'SH_Food_Donors_Dataset': {
   'tokens':['food','donors'], 'sheet':'Food Donors'},
 'SH_Food_Donations_Mock_Dataset': {
   'tokens':['food','donations'], 'sheet':'Food Donations'},
}
def norm(value):
    text=unicodedata.normalize('NFKD',str(value))
    text=''.join(ch for ch in text if not unicodedata.combining(ch))
    return re.sub(r'[^a-z0-9]+',' ',text.lower()).strip()

FILE_OVERRIDES = {
    # Example: 'SH_Food_Donations_Mock_Dataset': 'exact filename.xlsx'
}
excel_files=sorted(p for p in DATA_DIR.iterdir() if p.suffix.lower() in {'.xlsx','.xls'})
print('Excel files found:',len(excel_files))
for p in excel_files: print(' •',p.name)



Excel files found: 6
 • National-NGO-Organizations-Updated-Synthetic-Data-Disclosure.xlsx
 • SH-Donation-Allocations-Mock-Dataset.xlsx
 • SH-Organizations-Mock-Dataset-v2.xlsx
 • SH_Food_Donations_Mock_Dataset.xlsx
 • SH_Food_Donors_Dataset.xlsx
 • SH_Population_Service_and_Demand_Mock-Dataset.xlsx


## 2. Load the six primary tables

The expected primary sheet is loaded from each workbook. The three available Data Dictionary tabs are read only to clarify field descriptions; other workbook tabs are not treated as datasets.


In [ ]:
mapping={}
for dataset,spec in SPECS.items():
    override=FILE_OVERRIDES.get(dataset)
    if override:
        matches=[p for p in excel_files if p.name==override]
    else:
        tokens=spec['tokens']
        ranked=[(sum(token in norm(p.stem).split() for token in tokens),p) for p in excel_files]
        best=max((score for score,_ in ranked),default=0)
        matches=[p for score,p in ranked if score==best and best>0]
    if len(matches)!=1:
        raise ValueError(f'Could not uniquely match {dataset}. Candidates: {[p.name for p in matches]}. Set FILE_OVERRIDES.')
    mapping[dataset]=matches[0]

raw={}
source_definitions={}
for dataset,spec in SPECS.items():
    path=mapping[dataset]
    book=pd.ExcelFile(path)
    if spec['sheet'] not in book.sheet_names:
        raise ValueError(f"{path.name} does not have the expected primary sheet '{spec['sheet']}'. Sheets present: {book.sheet_names}")
    raw[dataset]=pd.read_excel(path,sheet_name=spec['sheet']).copy(deep=True)
    if dataset in {'SH_Food_Donors_Dataset','SH_Food_Donations_Mock_Dataset','SH Donation Allocations Mock Dataset'} and 'Data Dictionary' in book.sheet_names:

        # Use provided definitions as references; do not add this tab to the analyzed tables.
        dd=pd.read_excel(path,sheet_name='Data Dictionary')
        field_col=next((c for c in dd.columns if norm(c) in {'field','field name'}),None)
        meaning_col=next((c for c in dd.columns if norm(c) in {'definition caveat','description','description business meaning'}),None)
        if field_col is not None and meaning_col is not None:
            source_definitions[dataset]={str(row[field_col]):str(row[meaning_col]) for _,row in dd.iterrows() if pd.notna(row[field_col]) and pd.notna(row[meaning_col])}

table_summary=pd.DataFrame([{'Dataset':d,'Workbook':mapping[d].name,'Primary sheet':SPECS[d]['sheet'],
 'Rows':len(df),'Columns':len(df.columns)} for d,df in raw.items()])
display(table_summary)


,Dataset,Workbook,Primary sheet,Rows,Columns
0,National NGO Organizations Updated Synthetic-D...,National-NGO-Organizations-Updated-Synthetic-D...,Organizations,11287,23
1,SH Population Service and Demand Mock Dataset,SH_Population_Service_and_Demand_Mock-Dataset....,Population Service Demand,912,10
2,SH Donation Allocations Mock Dataset,SH-Donation-Allocations-Mock-Dataset.xlsx,Donation Allocations,4805,7
3,SH Organizations Mock Dataset v2,SH-Organizations-Mock-Dataset-v2.xlsx,Organizations,912,14
4,SH_Food_Donors_Dataset,SH_Food_Donors_Dataset.xlsx,Food Donors,227,9
5,SH_Food_Donations_Mock_Dataset,SH_Food_Donations_Mock_Dataset.xlsx,Food Donations,1200,7


## 3. Data dictionary

The dictionary includes observed type, examples, missing count, data origin, and source. Definitions from companion Data Dictionary tabs are used when available. For fields without a source definition, the description is a brief draft based on the field name. Origin labels: Real = source-recorded; Synthetic = simulated; Derived = calculated or inferred; Mixed = a combination or varies across records.


In [ ]:
KNOWN={
 'Donation ID':'Identifier for a donation record.',
 'Donor ID':'Identifier for a donor record.',
 'Organization ID':'Identifier assigned to an organization record.',
 'Recipient Organization ID':'Identifier for the organization receiving an allocation.',
 'Quantity in Pounds':'Recorded quantity of donated food, in pounds.',
 'Allocated Quantity in Pounds':'Recorded quantity allocated, in pounds.',
}
# Provenance notes are summarized from the source workbooks' dictionaries and methodology notes.
SOURCE_NOTES={
 'National NGO Organizations Updated Synthetic-Data Disclosure':'CRA charity records, Statistics Canada ODEF and ODHF, and organization sources listed in the workbook; some fields are synthetic or estimated.',
 'SH Population Service and Demand Mock Dataset':'SH Organizations source records, randomized user counts, and Second Harvest 2025 provincial baselines; demand measures are synthetic estimates.',
 'SH Donation Allocations Mock Dataset':'Food Donations and SH Organizations tables; allocation rows are simulated using report-based provincial totals.',
 'SH Organizations Mock Dataset v2':'National NGO source workbook and organization websites; some fields are simulated or derived estimates.',
 'SH_Food_Donors_Dataset':'Second Harvest Annual Report 2025, pages 43–46 for donor names; IDs, inferred categories, and assigned locations are mock or approximate.',
 'SH_Food_Donations_Mock_Dataset':'Second Harvest 2025 provincial meal figures; transaction-level rows and food categories are synthetic.',
}
ORIGINS={
 'National NGO Organizations Updated Synthetic-Data Disclosure':{
  'Organization ID':'Synthetic','Organization Name':'Real','Organization Type':'Mixed','Food-Related Service':'Mixed',
  'Province/Territory':'Real','City/Municipality':'Real','Street Address':'Real','Postal Code':'Real',
  'Latitude':'Mixed','Longitude':'Mixed','Coordinate Precision':'Derived','Users Served':'Synthetic',
  'User count 2025':'Synthetic','User Count Period':'Derived','Users Served Type':'Mixed','Food Demand Status':'Mixed',
  'Food Types Accepted':'Mixed','Storage/Preparation Capacity':'Mixed','Source Name':'Real','Source URL':'Real',
  'Data Year':'Mixed','Date Verified':'Mixed','Verification Status':'Mixed'},
 'SH Population Service and Demand Mock Dataset':{
  'Organization ID':'Derived','Postal Code':'Real','Users Served in 2025':'Synthetic',
  'Meals Provided by Second Harvest (Synthetic Estimate)':'Derived','Target Populations':'Derived',
  'Food Demand Status':'Derived','Unmet Demand (Estimated People; Synthetic)':'Synthetic',
  'Estimation Method':'Derived','Unmet Demand Estimation Rate (Synthetic)':'Synthetic','Province/Territory':'Real'},
 'SH Donation Allocations Mock Dataset':{
  'Donation ID':'Derived','Recipient Organization ID':'Derived','Province/Territory':'Derived',
  'Food Type':'Derived','Allocation Date':'Derived','Allocated Quantity in Pounds':'Synthetic',
  'Allocated Estimated Meals':'Derived'},
 'SH Organizations Mock Dataset v2':{
  'Organization ID':'Synthetic','Organization name':'Real','Organization type':'Real',
  'Food-related service or program':'Real','Official website':'Real','Operating status (simulated)':'Synthetic',
  'Province or territory':'Real','City or municipality':'Real','Street address':'Real','Postal code':'Real',
  'Latitude':'Mixed','Longitude':'Mixed','Rural/urban classification':'Derived','Food Types Accepted':'Synthetic'},
 'SH_Food_Donors_Dataset':{
  'Donor ID':'Synthetic','Donor Type':'Derived','Organization Name':'Real','Location':'Synthetic',
  'Longitude':'Synthetic','Latitude':'Synthetic','Report Category':'Derived','Location Data Status':'Derived','Source':'Real'},
 'SH_Food_Donations_Mock_Dataset':{
  'Donation ID':'Synthetic','Donor ID':'Synthetic','Province/Territory':'Derived','Food Type':'Synthetic',
  'Quantity in Pounds':'Mixed','Estimated Meals':'Derived','Donation Date':'Synthetic'},
}
def origin_for(dataset,field):
    return ORIGINS.get(dataset,{}).get(str(field),'Mixed')
def describe(field,dataset):
    if field in source_definitions.get(dataset,{}): return source_definitions[dataset][field]
    if field in KNOWN: return KNOWN[field]
    n=str(field).lower()
    if 'synthetic' in n or 'estimated' in n: return 'Synthetic or estimated measure, as labeled in the source.'
    if 'date' in n: return 'Date associated with the event or status named in the field.'
    if 'id' in n or n.endswith('key'): return 'Identifier as labeled in the source.'
    if any(x in n for x in ['province','territory','city','postal','address','location']): return 'Location field as labeled in the source.'
    if any(x in n for x in ['type','status','category','service','food']): return 'Category or description as labeled in the source.'
    return 'Meaning follows the source field name; confirm details and units from project documentation.'

province_code_domain = "AB | BC | MB | NB | NL | NS | NT | NU | ON | PE | QC | SK | YT"
standardized_province_fields = {
    ('National NGO Organizations Updated Synthetic-Data Disclosure', 'Province/Territory'),
    ('SH Population Service and Demand Mock Dataset', 'Province/Territory'),
    ('SH Donation Allocations Mock Dataset', 'Province/Territory'),
    ('SH Organizations Mock Dataset v2', 'Province or territory'),
    ('SH_Food_Donations_Mock_Dataset', 'Province/Territory'),
}

dict_rows=[]
for dataset,df in raw.items():
    for col in df.columns:
        s=df[col]; values=s.dropna().astype(str).unique().tolist()
        observed='; '.join(values[:15]) if len(values)<=15 else f'{len(values)} observed values; see source'
        dict_rows.append({'Dataset':dataset,'Primary sheet':SPECS[dataset]['sheet'],'Field':str(col),
          'Data type':str(s.dtype),'Description / business meaning':describe(col,dataset),
          'Description status':('Source dictionary' if col in source_definitions.get(dataset,{}) else 'PROVISIONAL — confirm with data owner'),
          'Allowed values / source domain':(f'Standardized codes used by this notebook: {province_code_domain}' if (dataset, str(col)) in standardized_province_fields else 'Not specified by source; observed values are not validated as allowed values.'),
          'Data origin':origin_for(dataset,col),'Source':SOURCE_NOTES.get(dataset,'Confirm source in workbook documentation.'),
          'Examples':' | '.join(map(str,s.dropna().head(3).tolist())),
          'Observed values (not necessarily complete allowed values)':observed,
          'Missing count':int(s.isna().sum()),
          'Missing-value rule':'Keep missing; do not replace with zero or infer a value.'})
data_dictionary=pd.DataFrame(dict_rows)
display(data_dictionary)

,Dataset,Primary sheet,Field,Data type,Description / business meaning,Description status,Allowed values / source domain,Data origin,Source,Examples,Observed values (not necessarily complete allowed values),Missing count,Missing-value rule
0,National NGO Organizations Updated Synthetic-D...,Organizations,Organization ID,str,Identifier assigned to an organization record.,PROVISIONAL — confirm with data owner,Not specified by source; observed values are n...,Synthetic,"CRA charity records, Statistics Canada ODEF an...",ORG000001 | ORG000002 | ORG000003,11287 observed values; see source,0,Keep missing; do not replace with zero or infe...
1,National NGO Organizations Updated Synthetic-D...,Organizations,Organization Name,str,Meaning follows the source field name; confirm...,PROVISIONAL — confirm with data owner,Not specified by source; observed values are n...,Real,"CRA charity records, Statistics Canada ODEF an...",Community Food Sharing Association | P.E.I. As...,11266 observed values; see source,0,Keep missing; do not replace with zero or infe...
2,National NGO Organizations Updated Synthetic-D...,Organizations,Organization Type,str,Category or description as labeled in the source.,PROVISIONAL — confirm with data owner,Not specified by source; observed values are n...,Mixed,"CRA charity records, Statistics Canada ODEF an...",Food bank network / food redistribution | Food...,233 observed values; see source,0,Keep missing; do not replace with zero or infe...
3,National NGO Organizations Updated Synthetic-D...,Organizations,Food-Related Service,str,Category or description as labeled in the source.,PROVISIONAL — confirm with data owner,Not specified by source; observed values are n...,Mixed,"CRA charity records, Statistics Canada ODEF an...",Bulk food collection and distribution to food ...,65 observed values; see source,0,Keep missing; do not replace with zero or infe...
4,National NGO Organizations Updated Synthetic-D...,Organizations,Province/Territory,str,Location field as labeled in the source.,PROVISIONAL — confirm with data owner,Standardized codes used by this notebook: AB |...,Real,"CRA charity records, Statistics Canada ODEF an...",Newfoundland and Labrador | Prince Edward Isla...,26 observed values; see source,0,Keep missing; do not replace with zero or infe...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
65,SH_Food_Donations_Mock_Dataset,Food Donations,Province/Territory,str,Province/territory distribution bucket used to...,Source dictionary,Standardized codes used by this notebook: AB |...,Derived,Second Harvest 2025 provincial meal figures; t...,Alberta | Ontario | New Brunswick,Alberta; Ontario; New Brunswick; British Colum...,0,Keep missing; do not replace with zero or infe...
66,SH_Food_Donations_Mock_Dataset,Food Donations,Food Type,str,Synthetic food category assigned to the transa...,Source dictionary,Not specified by source; observed values are n...,Synthetic,Second Harvest 2025 provincial meal figures; t...,Fresh produce | Meat and poultry | Bakery,Fresh produce; Meat and poultry; Bakery; Bever...,0,Keep missing; do not replace with zero or infe...
67,SH_Food_Donations_Mock_Dataset,Food Donations,Quantity in Pounds,int64,Synthetic transaction quantity. Provincial tot...,Source dictionary,Not specified by source; observed values are n...,Mixed,Second Harvest 2025 provincial meal figures; t...,91965 | 35625 | 18892,1193 observed values; see source,0,Keep missing; do not replace with zero or infe...
68,SH_Food_Donations_Mock_Dataset,Food Donations,Estimated Meals,int64,Calculated as Quantity in Pounds × 1. This is ...,Source dictionary,Not specified by source; observed values are n...,Derived,Second Harvest 2025 provincial meal figures; t...,91965 | 35625 | 18892,1193 observed values; see source,0,Keep missing; do not replace with zero or infe...


## 4. Profile the six tables

These summaries show missing values, exact duplicate rows, and field-level types and examples. ID uniqueness is checked only for fields intended to identify one record. Repeated foreign keys in allocation tables are normal and are not treated as duplicate records.


In [ ]:
table_profile=[]; field_profile=[]
for dataset,df in raw.items():
    table_profile.append({'Dataset':dataset,'Rows':len(df),'Columns':len(df.columns),
      'Missing cells':int(df.isna().sum().sum()),'Exact duplicate rows':int(df.duplicated().sum())})
    for col in df.columns:
        s=df[col]
        field_profile.append({'Dataset':dataset,'Field':str(col),'Data type':str(s.dtype),
          'Missing':int(s.isna().sum()),'Missing %':round(100*s.isna().mean(),2) if len(s) else 0,
          'Distinct non-missing':int(s.nunique()),'Examples':' | '.join(map(str,s.dropna().head(3).tolist()))})
table_profile=pd.DataFrame(table_profile); field_profile=pd.DataFrame(field_profile)
display(table_profile); display(field_profile)


,Dataset,Rows,Columns,Missing cells,Exact duplicate rows
0,National NGO Organizations Updated Synthetic-D...,11287,23,7168,0
1,SH Population Service and Demand Mock Dataset,912,10,332,0
2,SH Donation Allocations Mock Dataset,4805,7,0,0
3,SH Organizations Mock Dataset v2,912,14,285,0
4,SH_Food_Donors_Dataset,227,9,0,0
5,SH_Food_Donations_Mock_Dataset,1200,7,0,0


,Dataset,Field,Data type,Missing,Missing %,Distinct non-missing,Examples
0,National NGO Organizations Updated Synthetic-D...,Organization ID,str,0,0.0,11287,ORG000001 | ORG000002 | ORG000003
1,National NGO Organizations Updated Synthetic-D...,Organization Name,str,0,0.0,11266,Community Food Sharing Association | P.E.I. As...
2,National NGO Organizations Updated Synthetic-D...,Organization Type,str,0,0.0,233,Food bank network / food redistribution | Food...
3,National NGO Organizations Updated Synthetic-D...,Food-Related Service,str,0,0.0,65,Bulk food collection and distribution to food ...
4,National NGO Organizations Updated Synthetic-D...,Province/Territory,str,0,0.0,26,Newfoundland and Labrador | Prince Edward Isla...
...,...,...,...,...,...,...,...
65,SH_Food_Donations_Mock_Dataset,Province/Territory,str,0,0.0,13,Alberta | Ontario | New Brunswick
66,SH_Food_Donations_Mock_Dataset,Food Type,str,0,0.0,10,Fresh produce | Meat and poultry | Bakery
67,SH_Food_Donations_Mock_Dataset,Quantity in Pounds,int64,0,0.0,1193,91965 | 35625 | 18892
68,SH_Food_Donations_Mock_Dataset,Estimated Meals,int64,0,0.0,1193,91965 | 35625 | 18892


## 5. Clean copies, standardize City and check Province

The original tables are kept unchanged. Text is trimmed in working copies. City labels are standardized, and capitalization-only differences in Organization Type are automatically changed to the most common spelling and logged. Unrecognized values are preserved or flagged for review.

In [ ]:

cleaned={}; cleaning_rows=[]; issue_rows=[]; variant_rows=[]; category_case_rows=[]
UNIQUE_KEYS={
 'SH_Food_Donations_Mock_Dataset':['Donation ID'],
 'SH_Food_Donors_Dataset':['Donor ID'],
 'SH Organizations Mock Dataset v2':['Organization ID'],
 'SH Population Service and Demand Mock Dataset':['Organization ID'],
}
CITY_DATASET='SH Organizations Mock Dataset v2'
CITY_FIELD='City or municipality'
CITY_CANONICAL={
 'st john s':"St. John's",
 'montreal':'Montreal',
 'quebec':'Quebec',
 'chateauguay':'Chateauguay',
 'st catharines':'St. Catharines',
 'sault ste marie':'Sault Ste. Marie',
 'courice':'Courtice',
}
def standardize_city(value):
    if pd.isna(value): return value
    text=unicodedata.normalize('NFKD',str(value).strip())
    text=''.join(ch for ch in text if not unicodedata.combining(ch))
    text=re.sub(r'\s+',' ',text)
    key=norm(text)
    if key in CITY_CANONICAL: return CITY_CANONICAL[key]
    text=text.title()
    text=re.sub(r"(?<=\w)'([A-Z])",lambda m:"'"+m.group(1).lower(),text)
    text=re.sub(r'^St\.?\s+','St. ',text)
    text=re.sub(r'^Ste\.?\s+','Ste. ',text)
    return text

city_change_rows=[]
city_summary_rows=[]
province_summary_rows=[]
PROVINCE_CODES={
 'AB':'AB','ALBERTA':'AB',
 'BC':'BC','BRITISH COLUMBIA':'BC',
 'MB':'MB','MANITOBA':'MB',
 'NB':'NB','NEW BRUNSWICK':'NB',
 'NL':'NL','NEWFOUNDLAND AND LABRADOR':'NL',
 'NS':'NS','NOVA SCOTIA':'NS',
 'NT':'NT','NORTHWEST TERRITORIES':'NT',
 'NU':'NU','NUNAVUT':'NU',
 'ON':'ON','ONTARIO':'ON',
 'PE':'PE','PRINCE EDWARD ISLAND':'PE',
 'QC':'QC','QUEBEC':'QC',
 'SK':'SK','SASKATCHEWAN':'SK',
 'YT':'YT','YUKON':'YT',
}
NGO_DATASET='National NGO Organizations Updated Synthetic-Data Disclosure'
NGO_PROVINCE_FIELD='Province/Territory'
DONOR_DATASET='SH_Food_Donors_Dataset'
for dataset,original in raw.items():
    work=original.copy(deep=True); trimmed=blank=0
    coordinate_placeholders=coordinate_unparsed=0
    for col in work.select_dtypes(include=['object','string']).columns:
        before=work[col].astype('string'); after=before.str.strip()
        trimmed+=int((before.notna() & before.ne(after).fillna(False)).sum())
        blank_mask=after.eq('').fillna(False); blank+=int(blank_mask.sum())
        work[col]=after.mask(blank_mask,pd.NA)

    # Coordinate source fields contain numeric strings and the documented placeholder '..'.
    # Convert numeric strings in the working copy; keep raw unchanged and log any other token.
    for coordinate_field in ['Latitude','Longitude']:
        if coordinate_field in work.columns:
            coordinate_text=work[coordinate_field].astype('string').str.strip()
            placeholder=coordinate_text.eq('..').fillna(False)
            numeric=pd.to_numeric(coordinate_text.mask(placeholder),errors='coerce')
            unexpected=coordinate_text.notna() & ~placeholder & numeric.isna()
            coordinate_placeholders += int(placeholder.sum())
            coordinate_unparsed += int(unexpected.sum())
            if unexpected.any():
                issue_rows.append({'Dataset':dataset,'Field':coordinate_field,
                  'Issue':'Coordinate value is neither numeric nor the known placeholder ..',
                  'Count':int(unexpected.sum()),
                  'Examples':' | '.join(coordinate_text[unexpected].drop_duplicates().head(8).astype(str)),
                  'Review note':'Value retained for review; no guess or replacement applied.'})
                numeric=numeric.astype(object)
                numeric.loc[unexpected]=coordinate_text.loc[unexpected]
            work[coordinate_field]=numeric

    province_changed=0
    donor_province_unparsed=0

    province_field = None
    province_dataset_fields = {
        'SH Population Service and Demand Mock Dataset': 'Province/Territory',
        'SH Donation Allocations Mock Dataset': 'Province/Territory',
        'SH Organizations Mock Dataset v2': 'Province or territory',
        'SH_Food_Donations_Mock_Dataset': 'Province/Territory',
    }

    if dataset == NGO_DATASET and NGO_PROVINCE_FIELD in work.columns:
        province_field = NGO_PROVINCE_FIELD

    elif dataset in province_dataset_fields:
        province_field = province_dataset_fields[dataset]

    elif dataset == DONOR_DATASET and 'Location' in work.columns:
        parts = work['Location'].astype('string').str.rsplit(',', n=1)
        province_text = parts.str[-1].str.strip().str.upper()
        mapped = province_text.map(PROVINCE_CODES)
        has_location = work['Location'].notna()
        unresolved = has_location & mapped.isna()
        donor_province_unparsed = int(unresolved.sum())

        # Keep Location unchanged; create a separate province-code column.
        work['Province'] = mapped
        province_summary_rows.append({
            'Dataset': dataset,
            'Action': 'Extract province from Location',
            'Rows standardized': int((has_location & mapped.notna()).sum()),
            'Rows requiring review': donor_province_unparsed,
        })

        if unresolved.any():
            issue_rows.append({
                'Dataset': dataset,
                'Field': 'Location',
                'Issue': 'Province could not be parsed from Location',
                'Count': donor_province_unparsed,
                'Examples': ' | '.join(
                    work.loc[unresolved, 'Location'].drop_duplicates().head(8).astype(str)
                ),
                'Review note': 'Location was preserved; Province is missing for these rows.',
            })

    if province_field is not None and province_field in work.columns:
        before_province = work[province_field].astype('string').str.strip()
        mapped = before_province.str.upper().map(PROVINCE_CODES)
        known = before_province.notna() & mapped.notna()
        province_changed = int(
            (known & before_province.ne(mapped).fillna(False)).sum()
        )

        # Keep labels that the mapping does not recognize so they can be reviewed.
        work[province_field] = mapped.where(mapped.notna(), before_province)
        unknown = before_province.notna() & mapped.isna()

        if unknown.any():
            issue_rows.append({
                'Dataset': dataset,
                'Field': province_field,
                'Issue': 'Province label not recognized for standardization',
                'Count': int(unknown.sum()),
                'Examples': ' | '.join(
                    before_province[unknown].drop_duplicates().head(8).astype(str)
                ),
                'Review note': 'Original label retained; confirm before changing.',
            })

        province_summary_rows.append({
            'Dataset': dataset,
            'Action': 'Standardize labels to province codes',
            'Rows standardized': province_changed,
            'Rows requiring review': int(unknown.sum()),
        })

    org_type_case_changed=0
    if dataset==NGO_DATASET and 'Organization Type' in work.columns:
        # Standardize capitalization-only differences to the most common exact spelling.
        values=work['Organization Type'].astype('string')
        for _, indexes in values.groupby(values.str.casefold(), dropna=True).groups.items():
            group_values=values.loc[indexes].dropna()
            unique_values=group_values.unique().tolist()
            if len(unique_values)>1:
                canonical=group_values.value_counts().index[0]
                changed_mask=values.loc[indexes].notna() & values.loc[indexes].ne(canonical)
                changed=int(changed_mask.sum())
                if changed:
                    work.loc[indexes, 'Organization Type']=values.loc[indexes].mask(changed_mask, canonical)
                    org_type_case_changed += changed
                    category_case_rows.append({'Dataset':dataset,'Field':'Organization Type',
                      'Original value(s)':' | '.join(sorted(map(str, unique_values))),
                      'Standardized value':str(canonical),'Rows changed':changed,
                      'Rule':'Capitalization only; most common spelling retained.'})

    city_changed=0
    if dataset==CITY_DATASET and CITY_FIELD in work.columns:
        before=original[CITY_FIELD].astype('string')
        after=work[CITY_FIELD].map(standardize_city).astype('string')
        mask=before.notna() & before.ne(after).fillna(False)
        city_changed=int(mask.sum())
        work[CITY_FIELD]=after
        city_summary_rows.append({'Dataset':dataset,'City records':int(before.notna().sum()),
          'Distinct city labels before':int(before.nunique()),'Distinct city labels after':int(after.nunique()),
          'Rows changed':city_changed})
        # Record only the agreed special mappings, not every capitalization-only change.
        for key,canonical in CITY_CANONICAL.items():
            selected=before[before.map(lambda x: norm(x)==key if pd.notna(x) else False)]
            if len(selected):
                forms=sorted(selected.unique().tolist())
                if any(form!=canonical for form in forms):
                    city_change_rows.append({'Original form(s)':' | '.join(forms),'Standardized value':canonical,
                      'Rows affected':int(len(selected)),
                      'Change type':'Spelling correction' if key=='courice' else 'Accent/punctuation standardization'})
    cleaned[dataset]=work
    cleaning_rows.append({'Dataset':dataset,'Rows before':len(original),'Rows after':len(work),
      'Missing cells before':int(original.isna().sum().sum()),'Missing cells after':int(work.isna().sum().sum()),
      'Text values trimmed':trimmed,'Blank text set missing':blank,
      'Coordinate .. placeholders set to missing':coordinate_placeholders,
      'Other nonnumeric coordinates flagged':coordinate_unparsed,
      'City rows standardized':city_changed,'Organization Type values standardized':org_type_case_changed,
      'Province labels standardized':province_changed,
      'Donor provinces not parsed':donor_province_unparsed,
      'Duplicate rows before':int(original.duplicated().sum()),
      'Duplicate rows after':int(work.duplicated().sum())})
    for key in UNIQUE_KEYS.get(dataset,[]):
        if key in work:
            dup=work[key].notna() & work[key].duplicated(keep=False)
            if dup.any(): issue_rows.append({'Dataset':dataset,'Field':key,'Issue':'Repeated value in record ID','Count':int(dup.sum()),'Examples':' | '.join(map(str,work.loc[dup,key].drop_duplicates().head(8))),'Review note':'Check expected uniqueness; no rows removed.'})
    for col in work.columns:
        s=work[col]; name=str(col).lower()
        if pd.api.types.is_numeric_dtype(s) and any(t in name for t in ['quantity','pounds','meals','people','count','demand','served']):
            bad=s<0
            if bad.fillna(False).any(): issue_rows.append({'Dataset':dataset,'Field':str(col),'Issue':'Negative value in a potentially non-negative measure','Count':int(bad.sum()),'Examples':' | '.join(map(str,s[bad].head(8))),'Review note':'Check source rules; value retained.'})
        relevant=('city' in name or 'province' in name or 'territory' in name or 'type' in name or 'status' in name or 'food' in name or 'service' in name)
        if relevant and (s.dtype=='object' or pd.api.types.is_string_dtype(s)):
            vals=s.dropna().astype(str).unique().tolist()
            if len(vals)<=500:
                groups={}
                for v in vals:
                    k=norm(v)
                    if k: groups.setdefault(k,set()).add(v)
                for forms in groups.values():
                    if len(forms)>1:
                        variant_rows.append({'Dataset':dataset,'Field':str(col),'Values to review':' | '.join(sorted(forms)[:10]),'Reason':'Same words after ignoring case, accents, spaces, and punctuation.'})
# Add the derived donor field to the dictionary after the cleaned copies are created.
if DONOR_DATASET in cleaned and 'Province' in cleaned[DONOR_DATASET].columns:
    donor_province=cleaned[DONOR_DATASET]['Province']
    data_dictionary=pd.concat([data_dictionary,pd.DataFrame([{
      'Dataset':DONOR_DATASET,'Primary sheet':'Food Donors','Field':'Province',
      'Data type':str(donor_province.dtype),
      'Description / business meaning':'Province code parsed from Location; original Location is retained.',
      'Description status':'Derived field; definition follows implemented transformation.',
      'Allowed values / source domain':'Not specified by source; see standardized province-code mapping.',
      'Data origin':'Derived','Source':'Location field in the Food Donors sheet',
      'Examples':' | '.join(donor_province.dropna().astype(str).drop_duplicates().head(3)),
      'Observed values (not necessarily complete allowed values)':'; '.join(sorted(donor_province.dropna().astype(str).unique())),
      'Missing count':int(donor_province.isna().sum()),
      'Missing-value rule':'Keep missing; review locations that could not be parsed.'
    }])],ignore_index=True)

cleaning_log=pd.DataFrame(cleaning_rows)
issue_review=pd.DataFrame(issue_rows,columns=['Dataset','Field','Issue','Count','Examples','Review note'])
text_variants=pd.DataFrame(variant_rows,columns=['Dataset','Field','Values to review','Reason']).drop_duplicates()
city_standardization=pd.DataFrame(city_summary_rows)
city_change_log=pd.DataFrame(city_change_rows,columns=['Original form(s)','Standardized value','Rows affected','Change type'])
category_case_log=pd.DataFrame(category_case_rows,columns=['Dataset','Field','Original value(s)','Standardized value','Rows changed','Rule'])
display(cleaning_log)
display(city_standardization)
display(city_change_log)
display(category_case_log if len(category_case_log) else pd.DataFrame({'Result':['No capitalization-only category differences needed standardization.']}))
province_standardization=pd.DataFrame(province_summary_rows)
display(province_standardization)
display(issue_review if len(issue_review) else pd.DataFrame({'Result':['No repeated record IDs or negative measures were flagged.']}))
display(text_variants if len(text_variants) else pd.DataFrame({'Result':['No additional case/spacing/punctuation variants found in checked fields.']}))
display(data_dictionary[data_dictionary['Field'] == 'Province'])

,Dataset,Rows before,Rows after,Missing cells before,Missing cells after,Text values trimmed,Blank text set missing,Coordinate .. placeholders set to missing,Other nonnumeric coordinates flagged,City rows standardized,Organization Type values standardized,Province labels standardized,Donor provinces not parsed,Duplicate rows before,Duplicate rows after
0,National NGO Organizations Updated Synthetic-D...,11287,11287,7168,8750,14,0,1582,0,0,1,1287,0,0,0
1,SH Population Service and Demand Mock Dataset,912,912,332,332,0,0,0,0,0,0,912,0,0,0
2,SH Donation Allocations Mock Dataset,4805,4805,0,0,0,0,0,0,0,0,4805,0,0,0
3,SH Organizations Mock Dataset v2,912,912,285,399,2,0,114,0,660,0,912,0,0,0
4,SH_Food_Donors_Dataset,227,227,0,0,0,0,0,0,0,0,0,0,0,0
5,SH_Food_Donations_Mock_Dataset,1200,1200,0,0,0,0,0,0,0,0,1200,0,0,0


,Dataset,City records,Distinct city labels before,Distinct city labels after,Rows changed
0,SH Organizations Mock Dataset v2,912,453,374,660


,Original form(s),Standardized value,Rows affected,Change type
0,ST JOHN'S | ST. JOHN'S | St. John's,St. John's,19,Accent/punctuation standardization
1,MONTREAL | MONTRÉAL | Montreal | Montréal,Montreal,15,Accent/punctuation standardization
2,QUEBEC | QUÉBEC,Quebec,3,Accent/punctuation standardization
3,CHATEAUGUAY | CHÂTEAUGUAY,Chateauguay,2,Accent/punctuation standardization
4,ST CATHARINES | ST. CATHARINES | ST.CATHARINES,St. Catharines,6,Accent/punctuation standardization
5,SAULT STE MARIE | SAULT STE. MARIE,Sault Ste. Marie,2,Accent/punctuation standardization
6,COURICE,Courtice,1,Spelling correction


,Dataset,Field,Original value(s),Standardized value,Rows changed,Rule
0,National NGO Organizations Updated Synthetic-D...,Organization Type,Healthcare facility — Nursing and residential ...,Healthcare facility — Nursing and residential ...,1,Capitalization only; most common spelling reta...


,Dataset,Action,Rows standardized,Rows requiring review
0,National NGO Organizations Updated Synthetic-D...,Standardize labels to province codes,1287,0
1,SH Population Service and Demand Mock Dataset,Standardize labels to province codes,912,0
2,SH Donation Allocations Mock Dataset,Standardize labels to province codes,4805,0
3,SH Organizations Mock Dataset v2,Standardize labels to province codes,912,0
4,SH_Food_Donors_Dataset,Extract province from Location,227,0
5,SH_Food_Donations_Mock_Dataset,Standardize labels to province codes,1200,0


,Result
0,No repeated record IDs or negative measures we...


,Result
0,No additional case/spacing/punctuation variant...


,Dataset,Primary sheet,Field,Data type,Description / business meaning,Description status,Allowed values / source domain,Data origin,Source,Examples,Observed values (not necessarily complete allowed values),Missing count,Missing-value rule
70,SH_Food_Donors_Dataset,Food Donors,Province,str,Province code parsed from Location; original L...,Derived field; definition follows implemented ...,Not specified by source; see standardized prov...,Derived,Location field in the Food Donors sheet,SK | ON | AB,AB; BC; MB; NB; NL; NS; ON; PE; QC; SK; YT,0,Keep missing; review locations that could not ...


## 5a. NGO text placeholders and postal codes

Text placeholders in the National NGO list (`..`, `Not provided in NSPL 2025 source`, `page)`) are set to missing, and a reason column is added with the code *`Not provided by source`*. Postal codes in all tables are standardized to `A1A 1A1`. Incomplete or invalid postal codes are set to missing, flagged in `Postal Code Invalid`, and their original values are kept in the change log.

In [ ]:
# Applies to the cleaned working copies only; raw tables stay unchanged.
ORG_DATASET = 'SH Organizations Mock Dataset v2'
POP_DATASET = 'SH Population Service and Demand Mock Dataset'
ALLOC_DATASET = 'SH Donation Allocations Mock Dataset'
DONATION_DATASET = 'SH_Food_Donations_Mock_Dataset'

# 1) NGO text placeholders -> missing, with a reason code
PLACEHOLDER_FIELDS = {NGO_DATASET: ['City/Municipality', 'Street Address', 'Postal Code']}
REASON_CODE = 'Not provided by source'

def placeholder_mask(series):
    text = series.astype('string').str.strip()
    return (text.eq('..') | text.str.startswith('Not provided', na=False) | text.eq('page)')).fillna(False)

placeholder_rows = []
for dataset, fields in PLACEHOLDER_FIELDS.items():
    work = cleaned[dataset]
    for field in fields:
        mask = placeholder_mask(work[field])
        missing_before = int(work[field].isna().sum())
        examples = ' | '.join(work.loc[mask, field].astype(str).value_counts().index[:4])
        work[field] = work[field].mask(mask, pd.NA)
        reason_field = f'{field} Missing Reason'
        reason = pd.Series(pd.NA, index=work.index, dtype='string')
        reason[work[field].isna()] = REASON_CODE
        work.insert(work.columns.get_loc(field) + 1, reason_field, reason)
        placeholder_rows.append({'Dataset': dataset, 'Field': field, 'Placeholder values found': examples,
                                 'Placeholders set to missing': int(mask.sum()),
                                 'Missing before': missing_before, 'Missing after': int(work[field].isna().sum()),
                                 'Reason code added': REASON_CODE})
placeholder_log = pd.DataFrame(placeholder_rows)

# 2) Postal codes -> one format (A1A 1A1); invalid/truncated values -> missing + flag
POSTAL_FIELDS = {NGO_DATASET: 'Postal Code', ORG_DATASET: 'Postal code', POP_DATASET: 'Postal Code'}
POSTAL_PATTERN = r'[ABCEGHJ-NPRSTVXY]\d[A-Z]\d[A-Z]\d'   # Canada Post first-letter rules
postal_summary_rows = []; postal_change_rows = []
for dataset, field in POSTAL_FIELDS.items():
    work = cleaned[dataset]
    before = work[field].astype('string')
    compact = before.str.upper().str.replace(r'\s+', '', regex=True)
    valid = compact.str.fullmatch(POSTAL_PATTERN).fillna(False)
    formatted = (compact.str[:3] + ' ' + compact.str[3:]).where(valid)
    invalid = before.notna() & ~valid
    reformatted = valid & before.ne(formatted).fillna(False)
    id_field = 'Organization ID'
    for idx in work.index[invalid]:
        postal_change_rows.append({'Dataset': dataset, 'Organization ID': work.at[idx, id_field],
                                   'Field': field, 'Original value': str(before[idx]),
                                   'Action': 'Set to missing; Postal Code Invalid = Yes'})
    work[field] = formatted.astype('string')
    work.insert(work.columns.get_loc(field) + (2 if f'{field} Missing Reason' in work.columns else 1),
                'Postal Code Invalid', np.where(invalid, 'Yes', 'No'))
    postal_summary_rows.append({'Dataset': dataset, 'Field': field,
                                'Non-missing before': int(before.notna().sum()),
                                'Already in A1A 1A1 format': int((valid & ~reformatted).sum()),
                                'Reformatted to A1A 1A1': int(reformatted.sum()),
                                'Invalid set to missing': int(invalid.sum()),
                                'Non-missing after': int(work[field].notna().sum())})
postal_standardization = pd.DataFrame(postal_summary_rows)
postal_change_log = pd.DataFrame(postal_change_rows, columns=['Dataset', 'Organization ID', 'Field', 'Original value', 'Action'])

print('NGO text placeholders'); display(placeholder_log)
print('Postal code standardization'); display(postal_standardization)
print('Invalid postal codes (originals kept in this log)'); display(postal_change_log)


NGO text placeholders


,Dataset,Field,Placeholder values found,Placeholders set to missing,Missing before,Missing after,Reason code added
0,National NGO Organizations Updated Synthetic-D...,City/Municipality,..,5,16,21,Not provided by source
1,National NGO Organizations Updated Synthetic-D...,Street Address,Not provided in NSPL 2025 source; municipality...,1197,590,1787,Not provided by source
2,National NGO Organizations Updated Synthetic-D...,Postal Code,Not provided in NSPL 2025 source | .. | page),302,928,1230,Not provided by source


Postal code standardization


,Dataset,Field,Non-missing before,Already in A1A 1A1 format,Reformatted to A1A 1A1,Invalid set to missing,Non-missing after
0,National NGO Organizations Updated Synthetic-D...,Postal Code,10057,1727,8324,6,10051
1,SH Organizations Mock Dataset v2,Postal code,883,143,740,0,883
2,SH Population Service and Demand Mock Dataset,Postal Code,883,143,740,0,883


Invalid postal codes (originals kept in this log)


,Dataset,Organization ID,Field,Original value,Action
0,National NGO Organizations Updated Synthetic-D...,ORG007318,Postal Code,A0N1E,Set to missing; Postal Code Invalid = Yes
1,National NGO Organizations Updated Synthetic-D...,ORG007331,Postal Code,A0P1C,Set to missing; Postal Code Invalid = Yes
2,National NGO Organizations Updated Synthetic-D...,ORG007344,Postal Code,A0N1R,Set to missing; Postal Code Invalid = Yes
3,National NGO Organizations Updated Synthetic-D...,ORG007357,Postal Code,A2V1J,Set to missing; Postal Code Invalid = Yes
4,National NGO Organizations Updated Synthetic-D...,ORG007576,Postal Code,R2K4C,Set to missing; Postal Code Invalid = Yes
5,National NGO Organizations Updated Synthetic-D...,ORG008594,Postal Code,R0C1J,Set to missing; Postal Code Invalid = Yes


## 5b. Decision rules for flagged values

Each flagged value is handled by an explicit rule. Records are kept unless the rule says otherwise, missing values stay missing, and flag columns record the decision.

In [ ]:
ngo = cleaned[NGO_DATASET]; org = cleaned[ORG_DATASET]; pop = cleaned[POP_DATASET]
decision_rows = []
def add_rule(flag, dataset, field, before, rule, action, after):
    decision_rows.append({'Flagged value': flag, 'Dataset': dataset, 'Field': field, 'Count before': int(before),
                          'Rule': rule, 'Action taken': action, 'Count after / flag created': after})

# Rural/urban = "Unverified"
n = int(org['Rural/urban classification'].eq('Unverified').sum())
add_rule('Rural/urban = "Unverified"', ORG_DATASET, 'Rural/urban classification', n,
         'Keep as its own category, "Unverified".', 'No value changed; "Unverified" kept as a category.', f'{n} kept as "Unverified"')

# Operating status = "Requires verification"
n = int(org['Operating status (simulated)'].eq('Requires verification').sum())
add_rule('Operating status = "Requires verification"', ORG_DATASET, 'Operating status (simulated)', n,
         'Keep these records.', 'No rows removed.', f'{n} records kept')

# Users served missing
miss = pop['Users Served in 2025'].isna()
pop.insert(pop.columns.get_loc('Users Served in 2025') + 1, 'Users Served Missing', np.where(miss, 'Yes', 'No'))
add_rule('Users served missing', POP_DATASET, 'Users Served in 2025', miss.sum(),
         'Keep the row with users served missing label.', 'Row kept; value stays missing (not zero).',
         f'Users Served Missing = Yes for {int(miss.sum())} row(s)')

# Unmet demand unknown
unk = pop['Unmet Demand (Estimated People; Synthetic)'].isna()
pop.insert(pop.columns.get_loc('Unmet Demand (Estimated People; Synthetic)') + 1, 'Unmet Demand Status',
           np.where(unk, 'Unknown', 'Estimated'))
add_rule('Unmet demand unknown', POP_DATASET, 'Unmet Demand (Estimated People; Synthetic)', unk.sum(),
         'Keep as missing with the label "Unknown". Never set to zero.',
         'Values stay missing; Unmet Demand Status = "Unknown".', f'{int(unk.sum())} labelled "Unknown"; 0 set to zero')

# Duplicate postal codes (SH Organizations), counted after standardization
dup_postal = org['Postal code'].notna() & org['Postal code'].duplicated(keep=False)
add_rule('Duplicate postal codes (SH Organizations)', ORG_DATASET, 'Postal code', dup_postal.sum(),
         'Keep all of them.', 'No rows removed; Organization ID remains the unique key.', f'{int(dup_postal.sum())} rows kept')

# Possible duplicate organizations (NGO, same name and street address)
# Placeholder addresses were set to missing in 5a, so only real addresses are compared.
key = (ngo['Organization Name'].astype('string').str.strip().str.casefold() + ' | ' +
       ngo['Street Address'].astype('string').str.strip().str.casefold())
candidate = key.notna() & key.duplicated(keep=False)
linked_ids = set(org['Organization ID']).union(cleaned[ALLOC_DATASET]['Recipient Organization ID'])
ngo['Possible Duplicate'] = 'No'
duplicate_rows = []; drop_index = []
for k, idx in ngo[candidate].groupby(key[candidate]).groups.items():
    group = ngo.loc[idx]
    same = group['Organization Type'].nunique() == 1 and group['Food-Related Service'].nunique() == 1
    first_id = group['Organization ID'].iloc[0]
    for i, row in group.iterrows():
        if same and i != group.index[0] and row['Organization ID'] not in linked_ids:
            decision = f'Removed; duplicate of {first_id}'; drop_index.append(i)
        elif same and i != group.index[0]:
            decision = 'Kept and flagged; ID is linked to SH tables'; ngo.at[i, 'Possible Duplicate'] = 'Yes'
        elif same:
            decision = 'Kept (first ID)'
        else:
            decision = 'Kept and flagged; type or service differs'; ngo.at[i, 'Possible Duplicate'] = 'Yes'
        duplicate_rows.append({'Organization ID': row['Organization ID'], 'Organization Name': row['Organization Name'],
                               'Street Address': row['Street Address'], 'Organization Type': row['Organization Type'],
                               'Food-Related Service': row['Food-Related Service'],
                               'Type and service match': 'Yes' if same else 'No', 'Decision': decision})
duplicate_review = pd.DataFrame(duplicate_rows)
ngo.drop(index=drop_index, inplace=True)
cleaned[NGO_DATASET] = ngo
add_rule('Possible duplicate organizations (NGO, same name and address)', NGO_DATASET,
         'Organization Name + Street Address', candidate.sum(),
         'Review each pair. If the type and service also match, keep the first ID and log the removed one. '
         'If they differ, keep both and flag them.',
         f'{len(drop_index)} removed; {int((ngo["Possible Duplicate"] == "Yes").sum())} flagged',
         'Possible Duplicate = Yes/No')
print(f'Rows sharing name and address only because both addresses were placeholders are no longer matched: '
      f'{int(candidate.sum())} rows ({candidate.sum() // 2} pairs) compared on real addresses.')

# NGO text placeholders and truncated postal codes (applied in 5a)
for _, r in placeholder_log.iterrows():
    add_rule('NGO text placeholders ("..", "Not provided in NSPL 2025 source", "page)")', r['Dataset'], r['Field'],
             r['Placeholders set to missing'], 'Convert to missing and add a reason code: Not provided by source.',
             'Placeholders set to missing.', f'{r["Field"]} Missing Reason')
for _, r in postal_standardization.iterrows():
    if r['Invalid set to missing']:
        add_rule('Truncated postal codes (5 characters)', r['Dataset'], r['Field'], r['Invalid set to missing'],
                 'Set to missing, keep the original in the change log, and flag Postal Code Invalid.',
                 'Set to missing; originals in postal_change_log.', 'Postal Code Invalid = Yes/No')

# Missing coordinates
for dataset in [NGO_DATASET, ORG_DATASET]:
    work = cleaned[dataset]
    has = work['Latitude'].notna() & work['Longitude'].notna()
    work.insert(work.columns.get_loc('Longitude') + 1, 'Has Coordinates', np.where(has, 'Yes', 'No'))
    add_rule('Missing coordinates (after "..")', dataset, 'Latitude / Longitude', (~has).sum(),
             'Keep missing and add Has Coordinates (Yes/No).', 'Values stay missing.', f'Has Coordinates = No for {int((~has).sum())}')

# Users served randomized
ngo = cleaned[NGO_DATASET]
source = np.where(ngo['Users Served Type'].astype('string').str.startswith('Randomized', na=False), 'Randomized', 'Original estimate')
ngo.insert(ngo.columns.get_loc('Users Served Type') + 1, 'Users Served Source', source)
pop_source = pop['Organization ID'].map(ngo.set_index('Organization ID')['Users Served Source'])
pop.insert(pop.columns.get_loc('Users Served Missing') + 1, 'Users Served Source', pop_source)
for dataset, frame in [(NGO_DATASET, ngo), (POP_DATASET, pop)]:
    n = int(frame['Users Served Source'].eq('Randomized').sum())
    add_rule('Users served randomized (NGO "Users Served Type")', dataset, 'Users Served Source', n,
             'Keep the values, add Users Served Source (Original estimate / Randomized), and state it in the dictionary.',
             'Values kept; source label added.', f'{n} Randomized; {int(frame["Users Served Source"].eq("Original estimate").sum())} Original estimate')

decision_rules = pd.DataFrame(decision_rows)
print('Decision rules for flagged values'); display(decision_rules)
print('Possible duplicate review'); display(duplicate_review)


Rows sharing name and address only because both addresses were placeholders are no longer matched: 20 rows (10 pairs) compared on real addresses.
Decision rules for flagged values


,Flagged value,Dataset,Field,Count before,Rule,Action taken,Count after / flag created
0,"Rural/urban = ""Unverified""",SH Organizations Mock Dataset v2,Rural/urban classification,126,"Keep as its own category, ""Unverified"".","No value changed; ""Unverified"" kept as a categ...","126 kept as ""Unverified"""
1,"Operating status = ""Requires verification""",SH Organizations Mock Dataset v2,Operating status (simulated),91,Keep these records.,No rows removed.,91 records kept
2,Users served missing,SH Population Service and Demand Mock Dataset,Users Served in 2025,1,Keep the row with users served missing label.,Row kept; value stays missing (not zero).,Users Served Missing = Yes for 1 row(s)
3,Unmet demand unknown,SH Population Service and Demand Mock Dataset,Unmet Demand (Estimated People; Synthetic),151,"Keep as missing with the label ""Unknown"". Neve...","Values stay missing; Unmet Demand Status = ""Un...","151 labelled ""Unknown""; 0 set to zero"
4,Duplicate postal codes (SH Organizations),SH Organizations Mock Dataset v2,Postal code,102,Keep all of them.,No rows removed; Organization ID remains the u...,102 rows kept
5,"Possible duplicate organizations (NGO, same na...",National NGO Organizations Updated Synthetic-D...,Organization Name + Street Address,20,Review each pair. If the type and service also...,1 removed; 18 flagged,Possible Duplicate = Yes/No
6,"NGO text placeholders ("".."", ""Not provided in ...",National NGO Organizations Updated Synthetic-D...,City/Municipality,5,Convert to missing and add a reason code: Not ...,Placeholders set to missing.,City/Municipality Missing Reason
7,"NGO text placeholders ("".."", ""Not provided in ...",National NGO Organizations Updated Synthetic-D...,Street Address,1197,Convert to missing and add a reason code: Not ...,Placeholders set to missing.,Street Address Missing Reason
8,"NGO text placeholders ("".."", ""Not provided in ...",National NGO Organizations Updated Synthetic-D...,Postal Code,302,Convert to missing and add a reason code: Not ...,Placeholders set to missing.,Postal Code Missing Reason
9,Truncated postal codes (5 characters),National NGO Organizations Updated Synthetic-D...,Postal Code,6,"Set to missing, keep the original in the chang...",Set to missing; originals in postal_change_log.,Postal Code Invalid = Yes/No


Possible duplicate review


,Organization ID,Organization Name,Street Address,Organization Type,Food-Related Service,Type and service match,Decision
0,ORG000012,Daily Bread Food Bank,"191 New Toronto Street, Toronto, ON M8V 2E7",Food bank / food redistribution,Food bank services and food distribution throu...,No,Kept and flagged; type or service differs
1,ORG001020,Daily Bread Food Bank,"191 New Toronto Street, Toronto, ON M8V 2E7",Food bank / food redistribution,Food distribution through community agency net...,No,Kept and flagged; type or service differs
2,ORG000021,Edmonton's Food Bank,"11508 120 Street NW, Edmonton, AB T5G 2Y2",Food bank / food redistribution,Food assistance and distribution to partner ag...,No,Kept and flagged; type or service differs
3,ORG001015,Edmonton's Food Bank,"11508 120 Street NW, Edmonton, AB T5G 2Y2",Food bank / food redistribution,Food assistance and partner agency distribution,No,Kept and flagged; type or service differs
4,ORG000004,Feed NB / Food Depot Alimentaire,"330 MacNaughton Avenue, Moncton, NB E1H 2K1",Food bank network / food redistribution,"Food collection, storage and distribution to c...",No,Kept and flagged; type or service differs
5,ORG001018,Feed NB / Food Depot Alimentaire,"330 MacNaughton Avenue, Moncton, NB E1H 2K1",Food bank / food redistribution,Food collection and distribution,No,Kept and flagged; type or service differs
6,ORG000003,Feed Nova Scotia,"67 Wright Avenue, Dartmouth, NS B3B 1H2",Food bank network / food redistribution,Food collection and distribution to member age...,No,Kept and flagged; type or service differs
7,ORG001019,Feed Nova Scotia,"67 Wright Avenue, Dartmouth, NS B3B 1H2",Food bank / food redistribution,Food collection and distribution,No,Kept and flagged; type or service differs
8,ORG000015,Flemingdon Health Centre,"10 Gateway Boulevard, Toronto, ON M3C 3A1",Community health centre / food security program,"Community kitchen, community garden and food-s...",No,Kept and flagged; type or service differs
9,ORG001029,Flemingdon Health Centre,"10 Gateway Boulevard, Toronto, ON M3C 3A1",Community health / wellness centre,Community health and food-security programming,No,Kept and flagged; type or service differs


## 6. Check the links used in this project

These checks compare IDs across the six tables. A pass means the records in these files match on the selected keys; it does not verify real-world information. Repeated IDs in the allocation table are expected because a donation can be split across recipients.


In [ ]:
LINKS=[
 ('SH_Food_Donations_Mock_Dataset','Donor ID','SH_Food_Donors_Dataset','Donor ID','Donations to donors'),
 ('SH Donation Allocations Mock Dataset','Donation ID','SH_Food_Donations_Mock_Dataset','Donation ID','Allocations to donations'),
 ('SH Donation Allocations Mock Dataset','Recipient Organization ID','SH Organizations Mock Dataset v2','Organization ID','Allocations to organizations'),
 ('SH Population Service and Demand Mock Dataset','Organization ID','SH Organizations Mock Dataset v2','Organization ID','Service/demand to organizations'),
]
key_rows=[]
for child_ds,child_col,parent_ds,parent_col,label in LINKS:
    child=cleaned[child_ds][child_col] if child_col in cleaned[child_ds] else None
    parent=cleaned[parent_ds][parent_col] if parent_col in cleaned[parent_ds] else None
    if child is None or parent is None:
        key_rows.append({'Relationship':label,'Result':'Not checked','Details':'Expected key column not found.'}); continue
    parent_keys=set(parent.dropna().astype(str))
    unmatched=child.dropna()[~child.dropna().astype(str).isin(parent_keys)]
    key_rows.append({'Relationship':label,'Result':'Pass' if len(unmatched)==0 else 'Review',
      'Details':f'{len(unmatched)} non-empty child keys do not match a parent key.'})
integration_checks=pd.DataFrame(key_rows)
display(integration_checks)

don=cleaned['SH_Food_Donations_Mock_Dataset']; alloc=cleaned['SH Donation Allocations Mock Dataset']
reconciliation=pd.DataFrame()
if {'Donation ID','Quantity in Pounds'}.issubset(don.columns) and {'Donation ID','Allocated Quantity in Pounds'}.issubset(alloc.columns):
    d=don.groupby('Donation ID')['Quantity in Pounds'].sum(min_count=1).rename('Donated pounds')
    a=alloc.groupby('Donation ID')['Allocated Quantity in Pounds'].sum(min_count=1).rename('Allocated pounds')
    reconciliation=pd.concat([d,a],axis=1)
    reconciliation['Difference (donated - allocated)']=reconciliation['Donated pounds']-reconciliation['Allocated pounds']
    print('Donation totals compared:',len(reconciliation))
    print('IDs with a difference above 0.01 lb:',int((reconciliation.iloc[:,2].abs()>0.01).sum()))


,Relationship,Result,Details
0,Donations to donors,Pass,0 non-empty child keys do not match a parent key.
1,Allocations to donations,Pass,0 non-empty child keys do not match a parent key.
2,Allocations to organizations,Pass,0 non-empty child keys do not match a parent key.
3,Service/demand to organizations,Pass,0 non-empty child keys do not match a parent key.


Donation totals compared: 1200
IDs with a difference above 0.01 lb: 0


## 7. Final checks

This step updates the data dictionary using the cleaned tables, shows a short summary of key fields, and runs the final checks.

- `..` in latitude or longitude is treated as missing in the cleaned copy. Other nonnumeric coordinate values are flagged for review.
- `Date Verified` stays as text because some entries include notes, not just dates.
- The six tables stay separate. The notebook checks IDs and totals but does not combine the tables.
- Descriptions without a definition in the source file are marked `PROVISIONAL`.


In [ ]:
# Mark the standardized province-code domain only where this notebook applies that mapping.
province_codes = sorted(set(PROVINCE_CODES.values()))
source_domains = {
    (NGO_DATASET, NGO_PROVINCE_FIELD): " | ".join(province_codes),
    (DONOR_DATASET, "Province"): " | ".join(province_codes),
}
for row_index, row in data_dictionary.iterrows():
    domain = source_domains.get((row["Dataset"], row["Field"]))
    if domain:
        data_dictionary.at[row_index, "Allowed values / source domain"] = domain

# Refresh field type, examples, and missingness from the final cleaned copies.
# This makes the dictionary describe the deliverable dataset, not only the raw import.
for row_index, row in data_dictionary.iterrows():
    dataset, field = row["Dataset"], row["Field"]
    if dataset not in cleaned or field not in cleaned[dataset].columns:
        continue
    series = cleaned[dataset][field]
    nonmissing = series.dropna()
    distinct = nonmissing.nunique()
    data_dictionary.at[row_index, "Data type"] = str(series.dtype)
    data_dictionary.at[row_index, "Missing count"] = int(series.isna().sum())
    data_dictionary.at[row_index, "Examples"] = " | ".join(map(str, nonmissing.head(3).tolist()))
    observed = sorted(map(str, nonmissing.unique()), key=str.casefold) if distinct <= 15 else []
    data_dictionary.at[row_index, "Observed values (not necessarily complete allowed values)"] = (
        "; ".join(observed) if observed else (f"{distinct} observed values; see source" if distinct > 15 else "<none>")
    )

# A compact field list for the report body. Include fields only when present in the source schema.
KEY_FIELDS = {
    "National NGO Organizations Updated Synthetic-Data Disclosure": ["Organization ID", "Organization Name", "Province/Territory", "City/Municipality", "Users Served", "Food Demand Status"],
    "SH Population Service and Demand Mock Dataset": ["Organization ID", "Users Served in 2025", "Meals Provided by Second Harvest (Synthetic Estimate)", "Unmet Demand (Estimated People; Synthetic)", "Estimation Method", "Province/Territory"],
    "SH Donation Allocations Mock Dataset": ["Donation ID", "Recipient Organization ID", "Allocated Quantity in Pounds", "Allocated Estimated Meals", "Province/Territory"],
    "SH Organizations Mock Dataset v2": ["Organization ID", "Organization name", "Organization type", "City or municipality", "Province or territory", "Operating status (simulated)"],
    "SH_Food_Donors_Dataset": ["Donor ID", "Organization Name", "Donor Type", "Location", "Province", "Source"],
    "SH_Food_Donations_Mock_Dataset": ["Donation ID", "Donor ID", "Province/Territory", "Food Type", "Quantity in Pounds", "Estimated Meals", "Donation Date"],
}
summary_rows = []
for dataset, fields in KEY_FIELDS.items():
    for field in fields:
        matches = data_dictionary[(data_dictionary["Dataset"] == dataset) & (data_dictionary["Field"] == field)]
        if not matches.empty:
            row = matches.iloc[0]
            summary_rows.append({
                "Dataset": dataset, "Field": field, "Data type": row["Data type"],
                "Description / business meaning": row["Description / business meaning"],
                "Description status": row["Description status"],
                "Example value": row["Examples"],
                "Allowed values / source domain": row["Allowed values / source domain"],
                "Missing-value rule": row["Missing-value rule"],
            })
data_dictionary_summary = pd.DataFrame(summary_rows)
print("Report-body summary of key fields")
display(data_dictionary_summary)

# Use explicit field lists so text such as 'User Count Period' is never treated as a number.
NUMERIC_FIELDS = {
    "National NGO Organizations Updated Synthetic-Data Disclosure": ["Latitude", "Longitude", "Users Served", "User count 2025"],
    "SH Population Service and Demand Mock Dataset": ["Users Served in 2025", "Meals Provided by Second Harvest (Synthetic Estimate)", "Unmet Demand (Estimated People; Synthetic)", "Unmet Demand Estimation Rate (Synthetic)"],
    "SH Donation Allocations Mock Dataset": ["Allocated Quantity in Pounds", "Allocated Estimated Meals"],
    "SH Organizations Mock Dataset v2": ["Latitude", "Longitude"],
    "SH_Food_Donors_Dataset": ["Longitude", "Latitude"],
    "SH_Food_Donations_Mock_Dataset": ["Quantity in Pounds", "Estimated Meals"],
}
DATE_FIELDS = {
    "SH Donation Allocations Mock Dataset": ["Allocation Date"],
    "SH_Food_Donations_Mock_Dataset": ["Donation Date"],
}
TEXT_FIELDS_WITH_DATE_NOTES = {
    "National NGO Organizations Updated Synthetic-Data Disclosure": ["Date Verified"]
}
TEXT_FIELDS = {
    "National NGO Organizations Updated Synthetic-Data Disclosure": ["User Count Period", "Storage/Preparation Capacity", "Data Year", "Date Verified"],
}
type_rows = []
for dataset, frame in cleaned.items():
    for field in NUMERIC_FIELDS.get(dataset, []):
        if field not in frame.columns:
            type_rows.append({"Dataset":dataset,"Field":field,"Expected type":"numeric","Status":"NOT CHECKED — field missing"})
            continue
        values=frame[field].dropna()
        parsed=pd.to_numeric(values,errors="coerce")
        unparseable=int(parsed.isna().sum())
        dtype_ok=pd.api.types.is_numeric_dtype(frame[field])
        status="PASS" if dtype_ok and unparseable==0 else "REVIEW — values retained and flagged"
        type_rows.append({"Dataset":dataset,"Field":field,"Observed dtype":str(frame[field].dtype),
                          "Expected type":"numeric","Non-missing unparseable values":unparseable,"Status":status})
    for field in DATE_FIELDS.get(dataset, []):
        if field not in frame.columns:
            type_rows.append({"Dataset":dataset,"Field":field,"Expected type":"date","Status":"NOT CHECKED — field missing"})
            continue
        values=frame[field].dropna()
        parsed=pd.to_datetime(values,errors="coerce")
        unparseable=int(parsed.isna().sum())
        dtype_ok=pd.api.types.is_datetime64_any_dtype(frame[field])
        status="PASS" if dtype_ok and unparseable==0 else "REVIEW — values retained and flagged"
        type_rows.append({"Dataset":dataset,"Field":field,"Observed dtype":str(frame[field].dtype),
                          "Expected type":"date","Non-missing unparseable values":unparseable,"Status":status})
    for field in TEXT_FIELDS.get(dataset, []):
        if field not in frame.columns:
            continue
        series=frame[field]
        text_ok=(pd.api.types.is_object_dtype(series) or pd.api.types.is_string_dtype(series))
        interpretation=("text containing dates and explanatory notes; retained as text"
                        if field in TEXT_FIELDS_WITH_DATE_NOTES.get(dataset, []) else "text/category; retained as supplied")
        type_rows.append({"Dataset":dataset,"Field":field,"Observed dtype":str(series.dtype),
                          "Expected type":interpretation,"Status":"PASS" if text_ok else "REVIEW — confirm before conversion"})
data_type_review=pd.DataFrame(type_rows)
print("Type checks using explicit field expectations; no automatic type coercion")
display(data_type_review)

# Check coordinate bounds where the column names establish latitude/longitude semantics.
coordinate_rows = []
for dataset, frame in cleaned.items():
    for field in frame.columns:
        name = str(field).lower()
        if "latitude" in name or "longitude" in name:
            numeric = pd.to_numeric(frame[field], errors="coerce")
            lower, upper = (-90, 90) if "latitude" in name else (-180, 180)
            invalid_parse = int((frame[field].notna() & numeric.isna()).sum())
            out_of_range = int(((numeric < lower) | (numeric > upper)).sum())
            coordinate_rows.append({"Dataset": dataset, "Field": field, "Valid range": f"{lower} to {upper}",
                                    "Non-numeric non-missing": invalid_parse, "Out of range": out_of_range,
                                    "Status": "PASS" if invalid_parse == 0 and out_of_range == 0 else "REVIEW; values retained"})
coordinate_range_check = pd.DataFrame(coordinate_rows)
print("Coordinate range checks")
display(coordinate_range_check)

print("Integration decision: no source tables were joined or merged. Candidate links were checked by IDs; donation and allocation pounds were compared by Donation ID as a reconciliation only. No cross-source value was overwritten.")


Report-body summary of key fields


,Dataset,Field,Data type,Description / business meaning,Description status,Example value,Allowed values / source domain,Missing-value rule
0,National NGO Organizations Updated Synthetic-D...,Organization ID,string,Identifier assigned to an organization record.,PROVISIONAL — confirm with data owner,ORG000001 | ORG000002 | ORG000003,Not specified by source; observed values are n...,Keep missing; do not replace with zero or infe...
1,National NGO Organizations Updated Synthetic-D...,Organization Name,string,Meaning follows the source field name; confirm...,PROVISIONAL — confirm with data owner,Community Food Sharing Association | P.E.I. As...,Not specified by source; observed values are n...,Keep missing; do not replace with zero or infe...
2,National NGO Organizations Updated Synthetic-D...,Province/Territory,str,Location field as labeled in the source.,PROVISIONAL — confirm with data owner,NL | PE | NS,AB | BC | MB | NB | NL | NS | NT | NU | ON | P...,Keep missing; do not replace with zero or infe...
3,National NGO Organizations Updated Synthetic-D...,City/Municipality,string,Location field as labeled in the source.,PROVISIONAL — confirm with data owner,St. John's | Charlottetown | Dartmouth,Not specified by source; observed values are n...,Keep missing; do not replace with zero or infe...
4,National NGO Organizations Updated Synthetic-D...,Users Served,float64,Meaning follows the source field name; confirm...,PROVISIONAL — confirm with data owner,14422.0 | 5236.0 | 7327.0,Not specified by source; observed values are n...,Keep missing; do not replace with zero or infe...
5,National NGO Organizations Updated Synthetic-D...,Food Demand Status,string,Category or description as labeled in the source.,PROVISIONAL — confirm with data owner,Unknown unless documented separately | Unknown...,Not specified by source; observed values are n...,Keep missing; do not replace with zero or infe...
6,SH Population Service and Demand Mock Dataset,Organization ID,string,Identifier assigned to an organization record.,PROVISIONAL — confirm with data owner,ORG000986 | ORG000987 | ORG001012,Not specified by source; observed values are n...,Keep missing; do not replace with zero or infe...
7,SH Population Service and Demand Mock Dataset,Users Served in 2025,float64,Meaning follows the source field name; confirm...,PROVISIONAL — confirm with data owner,80.0 | 80.0 | 80.0,Not specified by source; observed values are n...,Keep missing; do not replace with zero or infe...
8,SH Population Service and Demand Mock Dataset,Meals Provided by Second Harvest (Synthetic Es...,int64,"Synthetic or estimated measure, as labeled in ...",PROVISIONAL — confirm with data owner,413 | 386 | 477,Not specified by source; observed values are n...,Keep missing; do not replace with zero or infe...
9,SH Population Service and Demand Mock Dataset,Unmet Demand (Estimated People; Synthetic),float64,"Synthetic or estimated measure, as labeled in ...",PROVISIONAL — confirm with data owner,6.0 | 6.0 | 21.0,Not specified by source; observed values are n...,Keep missing; do not replace with zero or infe...


Type checks using explicit field expectations; no automatic type coercion


,Dataset,Field,Observed dtype,Expected type,Non-missing unparseable values,Status
0,National NGO Organizations Updated Synthetic-D...,Latitude,Float64,numeric,0.0,PASS
1,National NGO Organizations Updated Synthetic-D...,Longitude,Float64,numeric,0.0,PASS
2,National NGO Organizations Updated Synthetic-D...,Users Served,float64,numeric,0.0,PASS
3,National NGO Organizations Updated Synthetic-D...,User count 2025,float64,numeric,0.0,PASS
4,National NGO Organizations Updated Synthetic-D...,User Count Period,string,text/category; retained as supplied,NaN,PASS
5,National NGO Organizations Updated Synthetic-D...,Storage/Preparation Capacity,string,text/category; retained as supplied,NaN,PASS
6,National NGO Organizations Updated Synthetic-D...,Data Year,string,text/category; retained as supplied,NaN,PASS
7,National NGO Organizations Updated Synthetic-D...,Date Verified,string,text containing dates and explanatory notes; r...,NaN,PASS
8,SH Population Service and Demand Mock Dataset,Users Served in 2025,float64,numeric,0.0,PASS
9,SH Population Service and Demand Mock Dataset,Meals Provided by Second Harvest (Synthetic Es...,int64,numeric,0.0,PASS


Coordinate range checks


,Dataset,Field,Valid range,Non-numeric non-missing,Out of range,Status
0,National NGO Organizations Updated Synthetic-D...,Latitude,-90 to 90,0,0,PASS
1,National NGO Organizations Updated Synthetic-D...,Longitude,-180 to 180,0,0,PASS
2,SH Organizations Mock Dataset v2,Latitude,-90 to 90,0,0,PASS
3,SH Organizations Mock Dataset v2,Longitude,-180 to 180,0,0,PASS
4,SH_Food_Donors_Dataset,Longitude,-180 to 180,0,0,PASS
5,SH_Food_Donors_Dataset,Latitude,-90 to 90,0,0,PASS


Integration decision: no source tables were joined or merged. Candidate links were checked by IDs; donation and allocation pounds were compared by Donation ID as a reconciliation only. No cross-source value was overwritten.


## 7a. Final data dictionary

One row per field in the cleaned tables, including fields added during cleaning. Columns: field name, plain data type, description, example value, allowed values, missing value rule, data origin and source.

In [ ]:
# Final data dictionary: one row per field in the cleaned tables (including fields added in 5a/5b).
# FIELD_INFO: description, plain data type, allowed values, missing-value rule.
# 'CATEGORIES' in allowed values = list the observed categories of the cleaned field.
PROV = 'AB | BC | MB | NB | NL | NS | NT | NU | ON | PE | QC | SK | YT'
FOOD_TYPES = 'Bakery | Beverages | Dairy and eggs | Fresh produce | Frozen foods | Meat and poultry | Mixed food products | Pantry staples | Prepared foods | Snacks and confectionery'
POSTAL = 'Canadian postal code format A1A 1A1'
LAT, LON = 'Decimal degrees, 41.6 to 83.2 (inside Canada)', 'Decimal degrees, -141.1 to -52.6 (inside Canada)'
YN = 'Yes | No'
NOT_ALLOWED = 'Not allowed; every record must have a value.'
KEEP = 'Keep missing; do not replace with zero or infer a value.'
CAT = 'CATEGORIES'
ORG_ID_DESC = 'Unique organization identifier (format ORG + 6 digits); join key between the NGO, SH Organizations, Population/Demand and Allocation tables.'

FIELD_INFO = {
 NGO_DATASET: {
  'Organization ID': ('Text (identifier)', ORG_ID_DESC, 'ORG000001 to ORG011287; unique', NOT_ALLOWED),
  'Organization Name': ('Text', 'Name of the organization as listed in the public source (CRA, StatCan ODEF/ODHF, NSPL or organization website).', 'Free text', NOT_ALLOWED),
  'Organization Type': ('Text (category)', 'Type of organization from the source, e.g. CRA charity category, educational or healthcare facility type, food bank.', 'Source categories (233 values)', NOT_ALLOWED),
  'Food-Related Service': ('Text', 'Food-related service or potential partner role; most national records are unverified potential partners.', 'Free text', NOT_ALLOWED),
  'Province/Territory': ('Text (code)', 'Province or territory where the organization is located, as a two-letter code.', PROV, NOT_ALLOWED),
  'City/Municipality': ('Text', 'City or municipality of the organization as recorded in the source.', 'Free text', 'Keep missing; reason recorded in City/Municipality Missing Reason.'),
  'City/Municipality Missing Reason': ('Text (category)', 'Why City/Municipality is missing. Blank when a city is recorded.', REASON_CODE, 'Blank when City/Municipality has a value.'),
  'Street Address': ('Text', 'Street or mailing address from the source.', 'Free text', 'Keep missing; reason recorded in Street Address Missing Reason.'),
  'Street Address Missing Reason': ('Text (category)', 'Why Street Address is missing. Blank when an address is recorded.', REASON_CODE, 'Blank when Street Address has a value.'),
  'Postal Code': ('Text', 'Canadian postal code, standardized to A1A 1A1.', POSTAL, 'Keep missing; see Postal Code Missing Reason and Postal Code Invalid.'),
  'Postal Code Missing Reason': ('Text (category)', 'Why Postal Code is missing because the source did not provide it. Blank when a postal code is recorded or when it was removed as invalid (see Postal Code Invalid).', REASON_CODE, 'Blank when Postal Code has a value or was invalid.'),
  'Postal Code Invalid': ('Text (flag)', 'Yes when the source postal code was incomplete or invalid (e.g. 5 characters) and was set to missing; original kept in the postal change log.', YN, NOT_ALLOWED),
  'Latitude': ('Decimal', 'Latitude of the organization; often approximate (see Coordinate Precision).', LAT, 'Keep missing; see Has Coordinates.'),
  'Longitude': ('Decimal', 'Longitude of the organization; often approximate (see Coordinate Precision).', LON, 'Keep missing; see Has Coordinates.'),
  'Has Coordinates': ('Text (flag)', 'Yes when both latitude and longitude are available.', YN, NOT_ALLOWED),
  'Coordinate Precision': ('Text (category)', 'How the coordinate was obtained (source-provided, approximate city-level, not available).', CAT, NOT_ALLOWED),
  'Users Served': ('Integer', 'Estimated people served per year. Synthetic: randomized for 10,000 records, proxy estimates for the rest (see Users Served Source).', 'Whole number >= 0', KEEP),
  'User count 2025': ('Integer', 'Same values as Users Served; kept for traceability to the source workbook.', 'Whole number >= 0', KEEP),
  'User Count Period': ('Text (category)', 'Period or basis of the user count.', CAT, NOT_ALLOWED),
  'Users Served Type': ('Text', 'Method used to produce Users Served (randomized value, HungerCount proxy, shelter beds x 4, etc.).', 'Free text', NOT_ALLOWED),
  'Users Served Source': ('Text (category)', 'Derived from Users Served Type: Randomized = value sampled from other records (not tied to the organization); Original estimate = organization-specific proxy estimate.', 'Original estimate | Randomized', NOT_ALLOWED),
  'Food Demand Status': ('Text (category)', 'Status of food demand information; all values are unverified proxies or potential-partner notes.', CAT, NOT_ALLOWED),
  'Food Types Accepted': ('Text (category)', 'Food types accepted; not reported for national records.', CAT, NOT_ALLOWED),
  'Storage/Preparation Capacity': ('Text', 'Storage or food preparation capacity; mostly not reported.', 'Free text', NOT_ALLOWED),
  'Source Name': ('Text (category)', 'Name of the public source of the record.', 'Source list (30 values)', NOT_ALLOWED),
  'Source URL': ('Text (URL)', 'Web address of the source.', 'Valid URL', NOT_ALLOWED),
  'Data Year': ('Text (category)', 'Year or version of the source data.', CAT, NOT_ALLOWED),
  'Date Verified': ('Text', 'Date the record was reviewed; some entries include a note, so kept as text.', 'YYYY-MM-DD, optionally followed by a note', NOT_ALLOWED),
  'Verification Status': ('Text (category)', 'Level of verification of name, address, coordinates and counts.', CAT, NOT_ALLOWED),
  'Possible Duplicate': ('Text (flag)', 'Yes when another record has the same name and street address but a different type or service; both kept.', YN, NOT_ALLOWED),
 },
 POP_DATASET: {
  'Organization ID': ('Text (identifier)', ORG_ID_DESC, 'Organization IDs in SH Organizations (912); unique', NOT_ALLOWED),
  'Postal Code': ('Text', 'Postal code of the partner organization, standardized to A1A 1A1.', POSTAL, KEEP),
  'Postal Code Invalid': ('Text (flag)', 'Yes when the source postal code was invalid and set to missing.', YN, NOT_ALLOWED),
  'Users Served in 2025': ('Integer', 'Estimated people served by the partner in 2025 (synthetic; copied from the national list, randomized for most partners).', 'Whole number >= 0', 'Keep missing; Users Served Missing = Yes.'),
  'Users Served Missing': ('Text (flag)', 'Yes when Users Served in 2025 is missing.', YN, NOT_ALLOWED),
  'Users Served Source': ('Text (category)', 'Origin of Users Served in 2025, linked from the national list by Organization ID.', 'Original estimate | Randomized', NOT_ALLOWED),
  'Meals Provided by Second Harvest (Synthetic Estimate)': ('Integer', 'Meals allocated to the partner in 2025; synthetic split of the provincial report baseline weighted by users served. Equals the sum of its allocated meals.', 'Whole number > 0', NOT_ALLOWED),
  'Target Populations': ('Text (category)', 'Groups the partner serves, inferred from name and service text; several groups separated by ";".', CAT, NOT_ALLOWED),
  'Food Demand Status': ('Text (category)', 'Whether an unmet-demand estimate is available.', CAT, NOT_ALLOWED),
  'Unmet Demand (Estimated People; Synthetic)': ('Integer', 'Estimated people whose food need is not met = Users Served in 2025 x gap rate (synthetic).', 'Whole number >= 0 and <= Users Served in 2025', 'Keep missing; Unmet Demand Status = Unknown. Never set to zero.'),
  'Unmet Demand Status': ('Text (category)', 'Estimated when an unmet-demand value exists; Unknown when it does not.', 'Estimated | Unknown', NOT_ALLOWED),
  'Estimation Method': ('Text', 'How meals, unmet demand and target populations were estimated.', 'Free text (2 values)', NOT_ALLOWED),
  'Unmet Demand Estimation Rate (Synthetic)': ('Decimal', 'Seeded random gap rate used to estimate unmet demand.', '0 to 1 (documented 0.05 to 0.36)', 'Keep missing when unmet demand is Unknown.'),
  'Province/Territory': ('Text (code)', 'Province or territory of the partner, as a two-letter code.', PROV, NOT_ALLOWED),
 },
 ALLOC_DATASET: {
  'Donation ID': ('Text (identifier)', 'Donation key matching Food Donations; one donation can appear on several rows, one per recipient.', 'D00001 to D01200', NOT_ALLOWED),
  'Recipient Organization ID': ('Text (identifier)', 'Partner organization receiving this share of the donation (simulated recipient).', 'Organization IDs in SH Organizations (912)', NOT_ALLOWED),
  'Province/Territory': ('Text (code)', 'Province of the donation and the recipient; always the same province.', PROV, NOT_ALLOWED),
  'Food Type': ('Text (category)', 'Food category of the parent donation.', FOOD_TYPES, NOT_ALLOWED),
  'Allocation Date': ('Date', 'Date of the parent donation (time removed).', '2025-01-01 to 2025-12-31', NOT_ALLOWED),
  'Allocated Quantity in Pounds': ('Integer', 'Pounds of the donation allocated to this recipient; rows for a donation add up to the donation total.', 'Whole number > 0', NOT_ALLOWED),
  'Allocated Estimated Meals': ('Integer', 'Allocated pounds x 1 (project proxy, not Second Harvest\'s official conversion).', 'Whole number > 0', NOT_ALLOWED),
 },
 ORG_DATASET: {
  'Organization ID': ('Text (identifier)', ORG_ID_DESC, 'Subset of National NGO IDs (912); unique', NOT_ALLOWED),
  'Organization name': ('Text', 'Name of the partner organization (same as in the National NGO list).', 'Free text; unique', NOT_ALLOWED),
  'Organization type': ('Text (category)', 'Type of organization (food bank, college, community health centre, charity, etc.).', 'Source categories (184 values)', NOT_ALLOWED),
  'Food-related service or program': ('Text (category)', 'Food-related service or program offered or potentially offered.', 'Source categories (33 values)', NOT_ALLOWED),
  'Official website': ('Text (URL)', 'Official website of the organization, or "Not found / requires verification" when no website was found.', 'Valid URL or "Not found / requires verification"', NOT_ALLOWED),
  'Operating status (simulated)': ('Text (category)', 'Simulated operating status of the partner.', 'Operating | Requires verification', NOT_ALLOWED),
  'Province or territory': ('Text (code)', 'Province or territory of the partner, as a two-letter code.', PROV, NOT_ALLOWED),
  'City or municipality': ('Text', 'City or municipality, standardized (title case, accents and punctuation unified).', 'Free text', KEEP),
  'Street address': ('Text', 'Street address of the partner.', 'Free text', KEEP),
  'Postal code': ('Text', 'Postal code, standardized to A1A 1A1.', POSTAL, KEEP),
  'Postal Code Invalid': ('Text (flag)', 'Yes when the source postal code was invalid and set to missing.', YN, NOT_ALLOWED),
  'Latitude': ('Decimal', 'Latitude of the partner; approximate for some records.', LAT, 'Keep missing; see Has Coordinates.'),
  'Longitude': ('Decimal', 'Longitude of the partner; approximate for some records.', LON, 'Keep missing; see Has Coordinates.'),
  'Has Coordinates': ('Text (flag)', 'Yes when both latitude and longitude are available.', YN, NOT_ALLOWED),
  'Rural/urban classification': ('Text (category)', 'Rural or urban location of the partner; "Unverified" kept as its own category.', CAT, NOT_ALLOWED),
  'Food Types Accepted': ('Text (list)', 'Food types the partner can accept, separated by ";" (simulated).', 'One or more of: ' + FOOD_TYPES, NOT_ALLOWED),
 },
 DONOR_DATASET: {
  'Donor ID': ('Text (identifier)', 'Synthetic unique donor identifier created for FoodBridge; not a Second Harvest ID.', 'FD0001 to FD0227; unique', NOT_ALLOWED),
  'Donor Type': ('Text (category)', 'Donor category inferred from the organization name.', CAT, NOT_ALLOWED),
  'Organization Name': ('Text', 'Donor name transcribed from the Second Harvest Annual Report 2025 (pp. 43-46).', 'Free text; unique', NOT_ALLOWED),
  'Location': ('Text', 'City and province assigned for modelling ("City, PR"); not necessarily the donor\'s headquarters.', '"City, PR"', NOT_ALLOWED),
  'Longitude': ('Decimal', 'Longitude of the assigned city centre; approximate.', LON, NOT_ALLOWED),
  'Latitude': ('Decimal', 'Latitude of the assigned city centre; approximate.', LAT, NOT_ALLOWED),
  'Report Category': ('Text (category)', 'Section of the annual report where the donor name appears.', CAT, NOT_ALLOWED),
  'Location Data Status': ('Text (category)', 'Flags the location as synthetic/approximate.', CAT, NOT_ALLOWED),
  'Source': ('Text', 'Source document of the donor name.', CAT, NOT_ALLOWED),
  'Province': ('Text (code)', 'Province code taken from Location; Location is kept unchanged.', PROV, 'Keep missing; review locations that could not be parsed.'),
 },
 DONATION_DATASET: {
  'Donation ID': ('Text (identifier)', 'Unique synthetic donation identifier.', 'D00001 to D01200; unique', NOT_ALLOWED),
  'Donor ID': ('Text (identifier)', 'Donor key matching Food Donors.', 'Donor IDs in Food Donors', NOT_ALLOWED),
  'Province/Territory': ('Text (code)', 'Province receiving the donation; calibrated to the report\'s provincial meal distribution.', PROV, NOT_ALLOWED),
  'Food Type': ('Text (category)', 'Synthetic food category of the donation.', FOOD_TYPES, NOT_ALLOWED),
  'Quantity in Pounds': ('Integer', 'Synthetic donation quantity; provincial totals calibrated to the 2025 report.', 'Whole number > 0', NOT_ALLOWED),
  'Estimated Meals': ('Integer', 'Quantity in Pounds x 1 (project proxy).', 'Whole number > 0', NOT_ALLOWED),
  'Donation Date': ('Date', 'Synthetic date of the donation.', '2025-01-01 to 2025-12-31', NOT_ALLOWED),
 },
}
NEW_FIELD_SOURCES = {
 'City/Municipality Missing Reason': 'City/Municipality', 'Street Address Missing Reason': 'Street Address',
 'Postal Code Missing Reason': 'Postal Code', 'Postal Code Invalid': 'Postal code field',
 'Has Coordinates': 'Latitude and Longitude', 'Users Served Source': 'Users Served Type (National NGO)',
 'Possible Duplicate': 'Organization Name, Street Address, Organization Type, Food-Related Service',
 'Users Served Missing': 'Users Served in 2025', 'Unmet Demand Status': 'Unmet Demand (Estimated People; Synthetic)',
 'Province': 'Location',
}
def plain_type_from_dtype(series):
    if pd.api.types.is_datetime64_any_dtype(series): return 'Date'
    if pd.api.types.is_integer_dtype(series): return 'Integer'
    if pd.api.types.is_float_dtype(series): return 'Decimal'
    return 'Text'

final_rows = []
for dataset, frame in cleaned.items():
    info = FIELD_INFO.get(dataset, {})
    for field in frame.columns:
        series = frame[field]; nonmissing = series.dropna()
        ftype, desc, allowed, rule = info.get(field, (plain_type_from_dtype(series), describe(field, dataset), 'Not specified', KEEP))
        if allowed == CAT:
            allowed = ' | '.join(sorted(map(str, nonmissing.unique()), key=str.casefold))
        is_new = field not in raw[dataset].columns
        origin = 'Derived' if is_new else origin_for(dataset, field)
        source = (f'Derived in this notebook from {NEW_FIELD_SOURCES.get(field, field)}' if is_new
                  else SOURCE_NOTES.get(dataset, 'Confirm source in workbook documentation.'))
        final_rows.append({'Dataset': dataset, 'Primary sheet': SPECS[dataset]['sheet'], 'Field name': str(field),
                           'Data type': ftype, 'Description / business meaning': desc,
                           'Example value': str(nonmissing.iloc[0]) if len(nonmissing) else '',
                           'Allowed values': allowed, 'Missing value rule': rule,
                           'Missing count': int(series.isna().sum()), 'Data origin': origin, 'Source': source,
                           'Field status': 'Added in cleaning' if is_new else 'Source field'})
data_dictionary = pd.DataFrame(final_rows)

undocumented = data_dictionary[data_dictionary['Allowed values'].eq('Not specified')]
print('Fields in the final dictionary:', len(data_dictionary), '| fields without a written definition:', len(undocumented))
if len(undocumented): display(undocumented[['Dataset', 'Field name']])

# Short table of key fields for the report body.
KEY_FIELDS[POP_DATASET] = KEY_FIELDS[POP_DATASET] + ['Users Served Source', 'Unmet Demand Status']
KEY_FIELDS[NGO_DATASET] = KEY_FIELDS[NGO_DATASET] + ['Users Served Source']
data_dictionary_summary = pd.concat([data_dictionary[(data_dictionary['Dataset'] == d) & data_dictionary['Field name'].isin(f)]
                                     for d, f in KEY_FIELDS.items()], ignore_index=True)[
    ['Dataset', 'Field name', 'Data type', 'Description / business meaning', 'Example value', 'Allowed values', 'Missing value rule', 'Data origin']]
print('Report-body summary of key fields'); display(data_dictionary_summary)
display(data_dictionary)


Fields in the final dictionary: 84 | fields without a written definition: 0
Report-body summary of key fields


,Dataset,Field name,Data type,Description / business meaning,Example value,Allowed values,Missing value rule,Data origin
0,National NGO Organizations Updated Synthetic-D...,Organization ID,Text (identifier),Unique organization identifier (format ORG + 6...,ORG000001,ORG000001 to ORG011287; unique,Not allowed; every record must have a value.,Synthetic
1,National NGO Organizations Updated Synthetic-D...,Organization Name,Text,Name of the organization as listed in the publ...,Community Food Sharing Association,Free text,Not allowed; every record must have a value.,Real
2,National NGO Organizations Updated Synthetic-D...,Province/Territory,Text (code),Province or territory where the organization i...,NL,AB | BC | MB | NB | NL | NS | NT | NU | ON | P...,Not allowed; every record must have a value.,Real
3,National NGO Organizations Updated Synthetic-D...,City/Municipality,Text,City or municipality of the organization as re...,St. John's,Free text,Keep missing; reason recorded in City/Municipa...,Real
4,National NGO Organizations Updated Synthetic-D...,Users Served,Integer,Estimated people served per year. Synthetic: r...,14422.0,Whole number >= 0,Keep missing; do not replace with zero or infe...,Synthetic
5,National NGO Organizations Updated Synthetic-D...,Users Served Source,Text (category),Derived from Users Served Type: Randomized = v...,Original estimate,Original estimate | Randomized,Not allowed; every record must have a value.,Derived
6,National NGO Organizations Updated Synthetic-D...,Food Demand Status,Text (category),Status of food demand information; all values ...,Unknown unless documented separately,Estimated higher potential need (population/se...,Not allowed; every record must have a value.,Mixed
7,SH Population Service and Demand Mock Dataset,Organization ID,Text (identifier),Unique organization identifier (format ORG + 6...,ORG000986,Organization IDs in SH Organizations (912); un...,Not allowed; every record must have a value.,Derived
8,SH Population Service and Demand Mock Dataset,Users Served in 2025,Integer,Estimated people served by the partner in 2025...,80.0,Whole number >= 0,Keep missing; Users Served Missing = Yes.,Synthetic
9,SH Population Service and Demand Mock Dataset,Users Served Source,Text (category),"Origin of Users Served in 2025, linked from th...",Original estimate,Original estimate | Randomized,Not allowed; every record must have a value.,Derived


,Dataset,Primary sheet,Field name,Data type,Description / business meaning,Example value,Allowed values,Missing value rule,Missing count,Data origin,Source,Field status
0,National NGO Organizations Updated Synthetic-D...,Organizations,Organization ID,Text (identifier),Unique organization identifier (format ORG + 6...,ORG000001,ORG000001 to ORG011287; unique,Not allowed; every record must have a value.,0,Synthetic,"CRA charity records, Statistics Canada ODEF an...",Source field
1,National NGO Organizations Updated Synthetic-D...,Organizations,Organization Name,Text,Name of the organization as listed in the publ...,Community Food Sharing Association,Free text,Not allowed; every record must have a value.,0,Real,"CRA charity records, Statistics Canada ODEF an...",Source field
2,National NGO Organizations Updated Synthetic-D...,Organizations,Organization Type,Text (category),"Type of organization from the source, e.g. CRA...",Food bank network / food redistribution,Source categories (233 values),Not allowed; every record must have a value.,0,Mixed,"CRA charity records, Statistics Canada ODEF an...",Source field
3,National NGO Organizations Updated Synthetic-D...,Organizations,Food-Related Service,Text,Food-related service or potential partner role...,Bulk food collection and distribution to food ...,Free text,Not allowed; every record must have a value.,0,Mixed,"CRA charity records, Statistics Canada ODEF an...",Source field
4,National NGO Organizations Updated Synthetic-D...,Organizations,Province/Territory,Text (code),Province or territory where the organization i...,NL,AB | BC | MB | NB | NL | NS | NT | NU | ON | P...,Not allowed; every record must have a value.,0,Real,"CRA charity records, Statistics Canada ODEF an...",Source field
...,...,...,...,...,...,...,...,...,...,...,...,...
79,SH_Food_Donations_Mock_Dataset,Food Donations,Province/Territory,Text (code),Province receiving the donation; calibrated to...,AB,AB | BC | MB | NB | NL | NS | NT | NU | ON | P...,Not allowed; every record must have a value.,0,Derived,Second Harvest 2025 provincial meal figures; t...,Source field
80,SH_Food_Donations_Mock_Dataset,Food Donations,Food Type,Text (category),Synthetic food category of the donation.,Fresh produce,Bakery | Beverages | Dairy and eggs | Fresh pr...,Not allowed; every record must have a value.,0,Synthetic,Second Harvest 2025 provincial meal figures; t...,Source field
81,SH_Food_Donations_Mock_Dataset,Food Donations,Quantity in Pounds,Integer,Synthetic donation quantity; provincial totals...,91965,Whole number > 0,Not allowed; every record must have a value.,0,Mixed,Second Harvest 2025 provincial meal figures; t...,Source field
82,SH_Food_Donations_Mock_Dataset,Food Donations,Estimated Meals,Integer,Quantity in Pounds x 1 (project proxy).,91965,Whole number > 0,Not allowed; every record must have a value.,0,Derived,Second Harvest 2025 provincial meal figures; t...,Source field


## 7b. Range and logic checks

These checks are validation evidence for the final dataset. Export runs only if every check passes.

In [ ]:
validation_rows = []
def add_check(check_type, check, dataset, rule, tested, failures):
    validation_rows.append({'Check type': check_type, 'Check': check, 'Dataset(s)': dataset, 'Rule': rule,
                            'Records tested': int(tested), 'Failures': int(failures),
                            'Status': 'Pass' if failures == 0 else 'Review'})

# Range: coordinates inside Canada
for dataset in [NGO_DATASET, ORG_DATASET, DONOR_DATASET]:
    f = cleaned[dataset]
    lat = pd.to_numeric(f['Latitude'], errors='coerce'); lon = pd.to_numeric(f['Longitude'], errors='coerce')
    tested = lat.notna() & lon.notna()
    outside = tested & ~(lat.between(41.6, 83.2) & lon.between(-141.1, -52.6))
    add_check('Range', 'Coordinates', dataset, 'Inside Canada (latitude 41.6 to 83.2, longitude -141.1 to -52.6)', tested.sum(), outside.sum())

# Range: unmet demand rate
rate = cleaned[POP_DATASET]['Unmet Demand Estimation Rate (Synthetic)'].dropna()
add_check('Range', 'Unmet demand rate', POP_DATASET, 'Between 0 and 1 (documented as 5% to 36%)', len(rate), (~rate.between(0, 1)).sum())

# Range: postal code format
for dataset, field in POSTAL_FIELDS.items():
    postal = cleaned[dataset][field].dropna().astype(str)
    add_check('Range', 'Postal codes', f'{dataset} ({field})', 'Canadian format (A1A 1A1)', len(postal),
              (~postal.str.fullmatch(r'[ABCEGHJ-NPRSTVXY]\d[A-Z] \d[A-Z]\d')).sum())

# Logic: unmet demand <= users served
p = cleaned[POP_DATASET]
both = p['Unmet Demand (Estimated People; Synthetic)'].notna() & p['Users Served in 2025'].notna()
add_check('Logic', 'Unmet demand vs users served', POP_DATASET, 'Unmet <= users served', both.sum(),
          (p.loc[both, 'Unmet Demand (Estimated People; Synthetic)'] > p.loc[both, 'Users Served in 2025']).sum())

# Logic: allocation province = donation province = recipient organization's province
a = cleaned[ALLOC_DATASET]
don_prov = a['Donation ID'].map(cleaned[DONATION_DATASET].set_index('Donation ID')['Province/Territory'])
org_prov = a['Recipient Organization ID'].map(cleaned[ORG_DATASET].set_index('Organization ID')['Province or territory'])
mismatch = a['Province/Territory'].ne(don_prov) | a['Province/Territory'].ne(org_prov) | don_prov.isna() | org_prov.isna()
add_check('Logic', 'Province consistency', f'{ALLOC_DATASET}, {DONATION_DATASET}, {ORG_DATASET}',
          "Allocation province = donation province = recipient organization's province", len(a), mismatch.sum())

# Logic: all 912 SH partners exist in the National NGO file with the same name
o = cleaned[ORG_DATASET]
ngo_names = cleaned[NGO_DATASET].set_index('Organization ID')['Organization Name']
matched = o['Organization ID'].map(ngo_names)
same_name = matched.astype('string').str.strip().str.casefold().eq(o['Organization name'].astype('string').str.strip().str.casefold()).fillna(False)
add_check('Logic', 'Subset link', f'{ORG_DATASET}, {NGO_DATASET}',
          'All 912 SH partners exist in the National NGO file with the same name', len(o), (~same_name).sum())

validation_checks = pd.DataFrame(validation_rows)
all_checks_pass = bool(validation_checks['Status'].eq('Pass').all())
display(validation_checks)
print('All validation checks passed.' if all_checks_pass else 'Some checks need review before export.')


,Check type,Check,Dataset(s),Rule,Records tested,Failures,Status
0,Range,Coordinates,National NGO Organizations Updated Synthetic-D...,"Inside Canada (latitude 41.6 to 83.2, longitud...",7679,0,Pass
1,Range,Coordinates,SH Organizations Mock Dataset v2,"Inside Canada (latitude 41.6 to 83.2, longitud...",732,0,Pass
2,Range,Coordinates,SH_Food_Donors_Dataset,"Inside Canada (latitude 41.6 to 83.2, longitud...",227,0,Pass
3,Range,Unmet demand rate,SH Population Service and Demand Mock Dataset,Between 0 and 1 (documented as 5% to 36%),761,0,Pass
4,Range,Postal codes,National NGO Organizations Updated Synthetic-D...,Canadian format (A1A 1A1),10050,0,Pass
5,Range,Postal codes,SH Organizations Mock Dataset v2 (Postal code),Canadian format (A1A 1A1),883,0,Pass
6,Range,Postal codes,SH Population Service and Demand Mock Dataset ...,Canadian format (A1A 1A1),883,0,Pass
7,Logic,Unmet demand vs users served,SH Population Service and Demand Mock Dataset,Unmet <= users served,760,0,Pass
8,Logic,Province consistency,"SH Donation Allocations Mock Dataset, SH_Food_...",Allocation province = donation province = reci...,4805,0,Pass
9,Logic,Subset link,"SH Organizations Mock Dataset v2, National NGO...",All 912 SH partners exist in the National NGO ...,912,0,Pass


All validation checks passed.


## 8. Export

Saves the cleaned tables (one Excel sheet per table), the data dictionary as CSV, and the cleaning and validation logs.

In [ ]:
if not all_checks_pass:
    raise RuntimeError('Export stopped: review the checks marked "Review" above.')

OUTPUT_DIR = DATA_DIR / 'foodbridge_outputs'
OUTPUT_DIR.mkdir(exist_ok=True)
SHEET_NAMES = {
    NGO_DATASET: 'National NGO', POP_DATASET: 'Population Demand', ALLOC_DATASET: 'Donation Allocations',
    ORG_DATASET: 'SH Organizations', DONOR_DATASET: 'Food Donors', DONATION_DATASET: 'Food Donations',
}
cleaned_path = OUTPUT_DIR / 'FoodBridge_Cleaned_Data.xlsx'
dictionary_path = OUTPUT_DIR / 'FoodBridge_Data_Dictionary.csv'
evidence_path = OUTPUT_DIR / 'FoodBridge_Cleaning_Evidence.xlsx'

# 1) Cleaned tables: one sheet per table.
with pd.ExcelWriter(cleaned_path, engine='openpyxl') as writer:
    for dataset, frame in cleaned.items():
        frame.to_excel(writer, sheet_name=SHEET_NAMES[dataset], index=False)

# 2) Data dictionary as CSV (utf-8-sig so Excel shows accents correctly).
data_dictionary.to_csv(dictionary_path, index=False, encoding='utf-8-sig')

# 3) Cleaning and validation logs used as evidence in the report.
EVIDENCE = {
    'Cleaning Log': cleaning_log, 'Placeholders': placeholder_log, 'Postal Standardization': postal_standardization,
    'Postal Change Log': postal_change_log, 'Decision Rules': decision_rules, 'Duplicate Review': duplicate_review,
    'City Standardization': city_standardization, 'City Change Log': city_change_log,
    'Province Standardization': province_standardization, 'Integration Checks': integration_checks,
    'Type Checks': data_type_review, 'Validation Checks': validation_checks, 'Dictionary Summary': data_dictionary_summary,
}
with pd.ExcelWriter(evidence_path, engine='openpyxl') as writer:
    for sheet, frame in EVIDENCE.items():
        frame.to_excel(writer, sheet_name=sheet, index=False)

# Row counts and file sizes (for the Final Output and Storage sections).
export_summary = pd.DataFrame([{'Table': SHEET_NAMES[d], 'Rows (raw)': len(raw[d]), 'Rows (cleaned)': len(f),
                                'Columns (raw)': raw[d].shape[1], 'Columns (cleaned)': f.shape[1],
                                'Unique key check': ('Pass' if d == ALLOC_DATASET or
                                                     f[UNIQUE_KEYS.get(d, ['Organization ID'])[0]].is_unique else 'Review')}
                               for d, f in cleaned.items()])
display(export_summary)
raw_size = sum(p.stat().st_size for p in mapping.values())
size_table = pd.DataFrame([{'File': 'Raw source workbooks (6)', 'Size (KB)': round(raw_size / 1024, 1)}] +
                          [{'File': p.name, 'Size (KB)': round(p.stat().st_size / 1024, 1)}
                           for p in [cleaned_path, dictionary_path, evidence_path]])
display(size_table)

try:
    from google.colab import files
    for p in [cleaned_path, dictionary_path, evidence_path]:
        files.download(str(p))
except ImportError:
    print('Files saved to', OUTPUT_DIR.resolve())


,Table,Rows (raw),Rows (cleaned),Columns (raw),Columns (cleaned),Unique key check
0,National NGO,11287,11286,23,30,Pass
1,Population Demand,912,912,10,14,Pass
2,Donation Allocations,4805,4805,7,7,Pass
3,SH Organizations,912,912,14,16,Pass
4,Food Donors,227,227,9,10,Pass
5,Food Donations,1200,1200,7,7,Pass


,File,Size (KB)
0,Raw source workbooks (6),2119.8
1,FoodBridge_Cleaned_Data.xlsx,2332.7
2,FoodBridge_Data_Dictionary.csv,38.9
3,FoodBridge_Cleaning_Evidence.xlsx,21.7


Files saved to foodbridge_outputs


## Notes

- The source tables remain separate; no merged analysis file is created. Raw files are never changed.
- City changes and the Organization Type capitalization correction are recorded above.
- Province and territory labels are standardized to two-letter codes across the six working tables. In Food Donors, the province code is derived from Location; the original Location column is retained.
- NGO text placeholders are set to missing with the reason code `Not provided by source`; postal codes use the `A1A 1A1` format in every table.
- Flagged values follow the decision rules in section 5b; flag columns (`Users Served Missing`, `Unmet Demand Status`, `Has Coordinates`, `Postal Code Invalid`, `Possible Duplicate`, `Users Served Source`) record each decision.
- Fields labeled synthetic, simulated, randomized, or estimated should not be presented as verified real-world activity.
- Outputs: `FoodBridge_Cleaned_Data.xlsx`, `FoodBridge_Data_Dictionary.csv`, `FoodBridge_Cleaning_Evidence.xlsx` in `foodbridge_outputs/`.
